In [ ]:
# | default_exp preprocessing.ocr.baidu_unlimited

In [ ]:
%load_ext autoreload
%autoreload 2

# Baidu Unlimited-OCR document parsing

> Recursively parse PDF pages, Office documents, and images into Markdown with
> a remote [Baidu Unlimited-OCR](https://github.com/baidu/Unlimited-OCR) vLLM
> service.

The default service is `172.27.74.16:7870`. The notebook uses the model's
required vLLM recipe: a literal `<image>` prompt prefix,
`skip_special_tokens=False`, and the per-request no-repeat n-gram parameters.
`layout_mode="plain"` sends each PDF page as an independent single-image
request. `layout_mode="pp-doclayout"` runs PP-DocLayout-V3 locally, recognizes
text/table/formula crops with Unlimited-OCR, preserves figure pixels, and
reconstructs the document in detector reading order.
Visible `.ppt`, `.pptx`, `.doc`, and `.docx` files are converted with
LibreOffice to sibling PDFs in their source folders before discovery; a
non-empty sibling PDF newer than its Office source is reused. Set
`LIBREOFFICE_BINARY` or pass `office_converter` when it is not on `PATH`.

Plain inputs produce UTF-8 Markdown below `.md_unlimited/` plus a sibling
`.unlimited.json` provenance file. Structured inputs produce Markdown,
schema-v2 `.layout.json`, and `.assets/` crops. While structured OCR is active,
`.partial.md`, `.partial.layout.json`, and a hidden checkpoint workspace are
refreshed after every region. Compatible work resumes automatically after
interruption; final Markdown, sidecar, and assets publish as one rollback-safe
bundle. Failed regions remain visible in a `partial` document instead of
discarding successful work.

Sidecars retain raw grounded model output. Markdown receives a cleaned form
with `<|ref|>` wrappers unwrapped and `<|det|>` coordinate boxes removed.
Existing outputs are skipped unless `overwrite=True`.

The service currently requires authentication. Put
`UNLIMITED_OCR_API_KEY=...` in the project-root `.env`, or pass `api_key=...`.
`UNLIMITED_OCR_BASE_URL` and `UNLIMITED_OCR_MODEL` can override the endpoint
and served model name. API keys are never written to output metadata.

In [ ]:
# | export
import asyncio
import base64
import copy
import fcntl
import json
import os
import re
import shutil
import signal
import subprocess
import warnings
from collections import Counter
from contextlib import asynccontextmanager, contextmanager
from dataclasses import dataclass
from datetime import datetime
from html import escape
from math import ceil, sqrt
from pathlib import Path
from tempfile import NamedTemporaryFile, TemporaryDirectory
from time import perf_counter
from typing import (
    Any,
    AsyncIterator,
    Awaitable,
    Callable,
    Iterator,
    Literal,
    Sequence,
    cast,
)
from urllib.parse import urlsplit

import httpx
import pymupdf
from dotenv import load_dotenv
from openai import AsyncOpenAI
from PIL import Image
from tqdm.auto import tqdm

from ribosome.preprocessing.ocr.utils import (
    ActivePageProgress as _ActivePageProgress,
)
from ribosome.preprocessing.ocr.utils import (
    OCRFolderExecution as _OCRFolderExecution,
)
from ribosome.preprocessing.ocr.utils import (
    OCRFolderPlan as _OCRFolderPlan,
)
from ribosome.preprocessing.ocr.utils import (
    find_project_root as _find_project_root,
)
from ribosome.preprocessing.ocr.utils import (
    ocr_folder_workflow as _ocr_folder_workflow,
)
from ribosome.preprocessing.ocr.utils import (
    positive_int as _positive_int,
)
from ribosome.preprocessing.ocr.utils import (
    resolve_output_dir_name as _resolve_output_dir_name,
)
from ribosome.preprocessing.ocr.utils import (
    resolve_root as _resolve_root,
)
from ribosome.preprocessing.ocr.utils import (
    running_in_notebook as _running_in_notebook,
)
from ribosome.preprocessing.ocr.utils import (
    value as _value,
)

In [ ]:
# | export
PROJ_ROOT = _find_project_root()
load_dotenv(PROJ_ROOT / ".env", override=False)

In [ ]:
# | export
@dataclass(frozen=True)
class UnlimitedOCRResult:
    """Outcome of parsing one PDF or image with Unlimited-OCR."""

    source_path: Path
    markdown_path: Path
    metadata_path: Path
    status: Literal["processed", "partial", "skipped", "failed"]
    pages_total: int = 0
    pages_completed: int = 0
    elapsed_s: float = 0.0
    error: str | None = None
    regions_total: int = 0
    regions_completed: int = 0
    regions_failed: int = 0


UnlimitedOCRLayoutMode = Literal["plain", "pp-doclayout"]


@dataclass(frozen=True)
class LayoutRegion:
    """One PP-DocLayout-V3 region in rendered-page pixel coordinates."""

    index: int
    label: str
    score: float
    bbox: tuple[int, int, int, int]
    task_type: Literal["text", "table", "formula", "figure"]


class _EmptyOCROutputError(ValueError):
    """Unlimited-OCR returned no usable text for an image."""


def _is_empty_ocr_error(error: object) -> bool:
    message = str(error or "")
    return (
        "Unlimited-OCR returned an empty response" in message
        or "Unlimited-OCR output is empty after post-processing" in message
    )


_SUPPORTED_SUFFIXES = frozenset(
    {".pdf", ".png", ".jpg", ".jpeg", ".webp", ".bmp"}
)
_OFFICE_SUFFIXES = frozenset({".ppt", ".pptx", ".doc", ".docx"})
_IMAGE_SUFFIXES = _SUPPORTED_SUFFIXES - {".pdf"}
_DEFAULT_BASE_URL = "172.27.74.16:7870"
_DEFAULT_MODEL = "baidu/Unlimited-OCR"
_DEFAULT_PROMPT = "<image>document parsing."
_DEFAULT_MAX_TOKENS = 8192
_DEFAULT_NGRAM_SIZE = 35
_DEFAULT_NGRAM_WINDOW = 128
_DEFAULT_MAX_DATA_URL_BYTES = 64 * 1024 * 1024
_DEFAULT_MAX_PAGE_PIXELS = 75_000_000
_DEFAULT_OFFICE_CONVERSION_TIMEOUT_S = 300.0
_DEFAULT_LAYOUT_MODEL = "PaddlePaddle/PP-DocLayoutV3_safetensors"
_DEFAULT_LAYOUT_MAX_TOKENS = 8192
_DEFAULT_LAYOUT_TEXT_MAX_TOKENS = 2048
_LAYOUT_RETRY_MAX_ASPECT_RATIO = 4.0
_LAYOUT_RETRY_MIN_SHORT_SIDE = 256
_LAYOUT_CHECKPOINT_SCHEMA_VERSION = 2
_LAYOUT_RESPONSE_VALIDATION_VERSION = 1
_LAYOUT_LABEL_TASKS = {
    "text": [
        "abstract",
        "algorithm",
        "aside_text",
        "content",
        "doc_title",
        "figure_title",
        "footer",
        "footnote",
        "formula_number",
        "header",
        "number",
        "paragraph_title",
        "reference",
        "reference_content",
        "seal",
        "text",
        "vertical_text",
        "vision_footnote",
    ],
    "table": ["table"],
    "formula": ["display_formula", "inline_formula"],
    "skip": ["chart", "footer_image", "header_image", "image"],
}
_LAYOUT_ASSET_TASKS = frozenset({"table", "formula", "figure"})

In [ ]:
# | export
def _metadata_path(markdown_path: Path) -> Path:
    return markdown_path.with_suffix(".unlimited.json")


def _visible_source_files(
    root: Path,
    output_dir_name: str,
    suffixes: frozenset[str],
) -> list[Path]:
    """Return matching files while pruning hidden and OCR output folders."""
    output_root = root / _resolve_output_dir_name(output_dir_name)
    sources: list[Path] = []
    for directory, directory_names, file_names in os.walk(root):
        directory_path = Path(directory)
        directory_names[:] = [
            name
            for name in directory_names
            if not name.startswith(".")
            and directory_path / name != output_root
        ]
        sources.extend(
            directory_path / name
            for name in file_names
            if not name.startswith("~$")
            and Path(name).suffix.casefold() in suffixes
        )
    sources.sort(
        key=lambda path: (
            path.relative_to(root).as_posix().casefold(),
            path.relative_to(root).as_posix(),
        )
    )
    return sources


def _office_sources(root: Path, output_dir_name: str) -> list[Path]:
    """Return Office sources after rejecting sibling-PDF name collisions."""
    sources = _visible_source_files(root, output_dir_name, _OFFICE_SUFFIXES)
    targets: dict[str, Path] = {}
    for source in sources:
        target = source.with_suffix(".pdf")
        collision_key = target.as_posix().casefold()
        if previous := targets.get(collision_key):
            raise ValueError(
                f"Office PDF collision: {previous} and {source} both map to {target}"
            )
        targets[collision_key] = source
    return sources


def _resolve_office_converter(converter: Path | str | None = None) -> str:
    """Resolve LibreOffice/soffice for Office-to-PDF conversion."""
    configured = (
        os.fspath(converter)
        if converter is not None
        else os.getenv("LIBREOFFICE_BINARY", "").strip()
    )
    candidates = (configured,) if configured else ("libreoffice", "soffice")
    for candidate in candidates:
        if resolved := shutil.which(candidate):
            return resolved
        candidate_path = Path(candidate).expanduser()
        if candidate_path.is_file():
            return str(candidate_path.resolve())
    raise FileNotFoundError(
        "LibreOffice executable not found. Install LibreOffice, set "
        "LIBREOFFICE_BINARY, or pass office_converter to ocr_folder()."
    )


def _office_pdf_is_current(source: Path, target: Path) -> bool:
    """Return whether a non-empty sibling PDF is at least as new as its source."""
    if not target.is_file():
        return False
    target_stat = target.stat()
    return (
        target_stat.st_size > 0
        and target_stat.st_mtime_ns >= source.stat().st_mtime_ns
    )


def _office_conversion_error(
    source: Path,
    completed: subprocess.CompletedProcess[str],
) -> RuntimeError:
    details = [
        line.strip()
        for output in (completed.stderr, completed.stdout)
        for line in output.splitlines()
        if line.strip()
    ]
    message = details[-1] if details else f"exit status {completed.returncode}"
    return RuntimeError(f"LibreOffice failed to convert {source}: {message}")


def _terminate_office_process(process: subprocess.Popen[str]) -> None:
    """Terminate a LibreOffice launcher and every child it spawned."""
    if os.name == "posix":
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    elif process.poll() is None:
        process.kill()


def _convert_office_to_pdf(
    source: Path,
    *,
    converter: str,
    timeout_s: float,
) -> Path:
    """Convert one Office file and atomically publish its sibling PDF."""
    target = source.with_suffix(".pdf")
    with TemporaryDirectory(
        prefix=".ribosome-office-pdf-",
        dir=source.parent,
    ) as temporary_directory:
        temporary_root = Path(temporary_directory)
        output_dir = temporary_root / "output"
        output_dir.mkdir()
        runtime_dir = temporary_root / "runtime"
        runtime_dir.mkdir(mode=0o700)
        config_dir = temporary_root / "config"
        config_dir.mkdir()
        cache_dir = temporary_root / "cache"
        cache_dir.mkdir()
        profile_uri = (temporary_root / "profile").resolve().as_uri()
        process_environment = os.environ.copy()
        process_environment.update(
            {
                "XDG_RUNTIME_DIR": str(runtime_dir),
                "XDG_CONFIG_HOME": str(config_dir),
                "XDG_CACHE_HOME": str(cache_dir),
                "SAL_USE_VCLPLUGIN": "svp",
            }
        )
        command = [
            converter,
            f"-env:UserInstallation={profile_uri}",
            "--headless",
            "--nologo",
            "--nodefault",
            "--nofirststartwizard",
            "--norestore",
            "--convert-to",
            "pdf",
            "--outdir",
            str(output_dir),
            str(source),
        ]
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            env=process_environment,
            start_new_session=os.name == "posix",
        )
        try:
            stdout, stderr = process.communicate(timeout=timeout_s)
        except subprocess.TimeoutExpired as error:
            _terminate_office_process(process)
            process.communicate()
            raise TimeoutError(
                f"LibreOffice timed out after {timeout_s:g}s converting {source}"
            ) from error
        except BaseException:
            _terminate_office_process(process)
            process.communicate()
            raise
        completed = subprocess.CompletedProcess(
            command,
            process.returncode,
            stdout,
            stderr,
        )
        if completed.returncode != 0:
            raise _office_conversion_error(source, completed)

        generated_pdfs = sorted(output_dir.glob("*.pdf"))
        if len(generated_pdfs) != 1 or generated_pdfs[0].stat().st_size == 0:
            raise RuntimeError(
                f"LibreOffice did not produce one non-empty PDF for {source}"
            )
        generated_pdf = generated_pdfs[0]
        try:
            with pymupdf.open(generated_pdf) as document:
                if document.needs_pass or document.page_count < 1:
                    raise ValueError("converted PDF is encrypted or empty")
        except (OSError, RuntimeError, ValueError) as error:
            raise RuntimeError(
                f"LibreOffice produced an invalid PDF for {source}: {error}"
            ) from error
        os.replace(generated_pdf, target)
    return target


async def _convert_office_sources_to_pdfs(
    root: Path,
    output_dir_name: str,
    *,
    converter: Path | str | None,
    timeout_s: float,
    show_progress: bool,
) -> list[Path]:
    """Create or reuse sibling PDFs for every visible Office source."""
    sources = _office_sources(root, output_dir_name)
    converted_pdfs: list[Path] = []
    resolved_converter: str | None = None
    for source in tqdm(
        sources,
        desc="Converting Office files to PDF",
        unit="file",
        dynamic_ncols=True,
        disable=not show_progress,
        leave=False,
    ):
        target = source.with_suffix(".pdf")
        if not _office_pdf_is_current(source, target):
            if resolved_converter is None:
                resolved_converter = _resolve_office_converter(converter)
            target = await asyncio.to_thread(
                _convert_office_to_pdf,
                source,
                converter=resolved_converter,
                timeout_s=timeout_s,
            )
            tqdm.write(f"Converted Office file: {source} -> {target}")
        converted_pdfs.append(target)
    return converted_pdfs


def _ocr_jobs(
    root: Path,
    output_dir_name: str,
) -> list[tuple[Path, Path, Path]]:
    """Return deterministic jobs while pruning hidden source folders."""
    output_root = root / _resolve_output_dir_name(output_dir_name)
    sources = _visible_source_files(root, output_dir_name, _SUPPORTED_SUFFIXES)

    jobs: list[tuple[Path, Path, Path]] = []
    targets: dict[str, Path] = {}
    for source_path in sources:
        relative = source_path.relative_to(root).with_suffix(".md")
        markdown_path = output_root / relative
        collision_key = markdown_path.as_posix().casefold()
        if previous := targets.get(collision_key):
            raise ValueError(
                f"OCR output collision: {previous} and {source_path} both map to "
                f"{markdown_path}"
            )
        targets[collision_key] = source_path
        jobs.append((source_path, markdown_path, _metadata_path(markdown_path)))
    return jobs

In [ ]:
# | export
def _resolve_base_url(base_url: str | None) -> str:
    resolved = (
        base_url.strip()
        if base_url is not None
        else os.getenv("UNLIMITED_OCR_BASE_URL", "").strip() or _DEFAULT_BASE_URL
    )
    if not resolved:
        raise ValueError("Unlimited-OCR base URL must not be empty")
    if "://" not in resolved:
        resolved = f"http://{resolved}"
    resolved = resolved.rstrip("/")
    parsed = urlsplit(resolved)
    if (
        parsed.scheme not in {"http", "https"}
        or not parsed.hostname
        or parsed.username is not None
        or parsed.password is not None
        or parsed.query
        or parsed.fragment
    ):
        raise ValueError(
            "Unlimited-OCR base URL must be an HTTP(S) endpoint without "
            "credentials, query, or fragment"
        )
    if not parsed.path:
        resolved += "/v1"
    elif not parsed.path.rstrip("/").endswith("/v1"):
        resolved += "/v1"
    return resolved


def _resolve_api_key(api_key: str | None) -> str:
    if api_key is not None:
        if not api_key.strip():
            raise ValueError("api_key must not be empty")
        return api_key.strip()
    return os.getenv("UNLIMITED_OCR_API_KEY", "").strip() or "EMPTY"


def _resolve_model(model: str | None) -> str:
    if model is not None:
        if not model.strip():
            raise ValueError("model must not be empty")
        return model.strip()
    return os.getenv("UNLIMITED_OCR_MODEL", "").strip() or _DEFAULT_MODEL


def _resolve_layout_mode(
    layout_mode: UnlimitedOCRLayoutMode | str,
) -> UnlimitedOCRLayoutMode:
    normalized = layout_mode.strip().casefold()
    if normalized not in {"plain", "pp-doclayout"}:
        raise ValueError("layout_mode must be 'plain' or 'pp-doclayout'")
    return cast(UnlimitedOCRLayoutMode, normalized)


def _resolve_prompt(prompt: str) -> str:
    normalized = prompt.strip()
    if not normalized.startswith("<image>"):
        raise ValueError("Unlimited-OCR prompt must begin with the literal <image>")
    return normalized


def create_pp_doclayout_detector(
    *,
    model_name: str = _DEFAULT_LAYOUT_MODEL,
    threshold: float = 0.3,
    device: str | None = None,
) -> Any:
    """Load and start the official GLM-OCR PP-DocLayout-V3 detector."""
    if not model_name.strip():
        raise ValueError("layout_model must not be empty")
    if not 0 <= threshold <= 1:
        raise ValueError("layout_threshold must be between 0 and 1")
    try:
        from glmocr.config import LayoutConfig
        from glmocr.layout import PPDocLayoutDetector
    except ImportError as error:
        raise RuntimeError(
            "PP-DocLayout-V3 requires the optional OCR layout runtime; "
            "run `uv sync --extra ocr-layout`"
        ) from error

    config = LayoutConfig(
        model_dir=model_name.strip(),
        threshold=threshold,
        batch_size=1,
        device=device,
        label_task_mapping=_LAYOUT_LABEL_TASKS,
    )
    detector = PPDocLayoutDetector(config)
    detector.start()
    return detector


def _create_client(
    base_url: str | None,
    api_key: str | None,
    request_timeout_s: float,
) -> AsyncOpenAI:
    """Create a proxy-free client suitable for a LAN vLLM endpoint."""
    resolved_base_url = _resolve_base_url(base_url)
    http_client = httpx.AsyncClient(
        timeout=request_timeout_s,
        trust_env=False,
    )
    return AsyncOpenAI(
        api_key=_resolve_api_key(api_key),
        base_url=resolved_base_url,
        timeout=request_timeout_s,
        max_retries=0,
        http_client=http_client,
    )


async def list_served_models(
    *,
    base_url: str | None = None,
    api_key: str | None = None,
    request_timeout_s: float = 30,
) -> list[str]:
    """Return model IDs advertised by the remote OpenAI-compatible service."""
    if request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be greater than zero")
    client = _create_client(base_url, api_key, request_timeout_s)
    try:
        response = await client.models.list()
        return [model.id for model in response.data]
    finally:
        await client.close()

In [ ]:
# | export
def _load_image_pixmap(path: Path) -> pymupdf.Pixmap:
    """Load a supported image at native resolution on an opaque white canvas."""
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", Image.DecompressionBombWarning)
            with Image.open(path) as image:
                image.load()
                rgba = image.convert("RGBA")
                background = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
                rgb = Image.alpha_composite(background, rgba).convert("RGB")
    except (Image.DecompressionBombWarning, Image.DecompressionBombError) as error:
        raise ValueError(
            f"Image exceeds Pillow's {Image.MAX_IMAGE_PIXELS:,}-pixel safety "
            f"limit: {path}"
        ) from error
    except Exception as error:
        raise ValueError(f"Could not decode image {path}: {error}") from error
    return pymupdf.Pixmap(
        pymupdf.csRGB,
        rgb.width,
        rgb.height,
        rgb.tobytes(),
        False,
    )


def _pdf_page_dimensions(page: pymupdf.Page, dpi: int) -> tuple[int, int]:
    """Return conservative rendered dimensions without allocating a pixmap."""
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    return (
        max(1, ceil(page.rect.width * dpi / 72)),
        max(1, ceil(page.rect.height * dpi / 72)),
    )


def _effective_pdf_page_dpi(
    page: pymupdf.Page,
    requested_dpi: int,
    max_page_pixels: int = _DEFAULT_MAX_PAGE_PIXELS,
) -> int:
    """Cap one PDF page's DPI so its rendered pixel count stays bounded."""
    if max_page_pixels <= 0:
        raise ValueError("max_page_pixels must be greater than zero")
    width, height = _pdf_page_dimensions(page, requested_dpi)
    requested_pixels = width * height
    if requested_pixels <= max_page_pixels:
        return requested_dpi

    effective_dpi = max(
        1,
        int(requested_dpi * sqrt(max_page_pixels / requested_pixels)),
    )
    while effective_dpi > 1:
        width, height = _pdf_page_dimensions(page, effective_dpi)
        if width * height <= max_page_pixels:
            break
        effective_dpi -= 1
    return effective_dpi


def _load_pdf_page(
    document: pymupdf.Document,
    page_index: int,
) -> tuple[pymupdf.Page, tuple[str, ...]]:
    """Load a page while capturing recoverable annotation diagnostics."""
    display_errors = bool(pymupdf.TOOLS.mupdf_display_errors())
    display_warnings = bool(pymupdf.TOOLS.mupdf_display_warnings())
    pymupdf.TOOLS.reset_mupdf_warnings()
    try:
        pymupdf.TOOLS.mupdf_display_errors(False)
        pymupdf.TOOLS.mupdf_display_warnings(False)
        page = document.load_page(page_index)
        load_warnings = tuple(
            dict.fromkeys(
                line.strip()
                for line in pymupdf.TOOLS.mupdf_warnings().splitlines()
                if line.strip()
            )
        )
    finally:
        pymupdf.TOOLS.reset_mupdf_warnings()
        pymupdf.TOOLS.mupdf_display_errors(display_errors)
        pymupdf.TOOLS.mupdf_display_warnings(display_warnings)
    return page, load_warnings


def _render_pdf_page(
    page: pymupdf.Page,
    *,
    dpi: int,
    max_page_pixels: int = _DEFAULT_MAX_PAGE_PIXELS,
) -> tuple[pymupdf.Pixmap, int, tuple[str, ...]]:
    """Render base page content and capture non-fatal MuPDF diagnostics."""
    effective_dpi = _effective_pdf_page_dpi(page, dpi, max_page_pixels)
    display_errors = bool(pymupdf.TOOLS.mupdf_display_errors())
    display_warnings = bool(pymupdf.TOOLS.mupdf_display_warnings())
    pymupdf.TOOLS.reset_mupdf_warnings()
    try:
        pymupdf.TOOLS.mupdf_display_errors(False)
        pymupdf.TOOLS.mupdf_display_warnings(False)
        pixmap = page.get_pixmap(
            dpi=effective_dpi,
            alpha=False,
            annots=False,
        )
        render_warnings = tuple(
            dict.fromkeys(
                line.strip()
                for line in pymupdf.TOOLS.mupdf_warnings().splitlines()
                if line.strip()
            )
        )
    finally:
        pymupdf.TOOLS.reset_mupdf_warnings()
        pymupdf.TOOLS.mupdf_display_errors(display_errors)
        pymupdf.TOOLS.mupdf_display_warnings(display_warnings)
    return pixmap, effective_dpi, render_warnings


def _pixmap_to_pil(pixmap: pymupdf.Pixmap) -> Image.Image:
    mode = "RGBA" if pixmap.alpha else ("L" if pixmap.n == 1 else "RGB")
    return Image.frombytes(
        mode,
        (pixmap.width, pixmap.height),
        pixmap.samples,
    ).convert("RGB")


def _pil_to_pixmap(image: Image.Image) -> pymupdf.Pixmap:
    rgb = image.convert("RGB")
    return pymupdf.Pixmap(
        pymupdf.csRGB,
        rgb.width,
        rgb.height,
        rgb.tobytes(),
        False,
    )


def _crop_pil_image(
    image: Image.Image,
    bbox: tuple[int, int, int, int],
) -> Image.Image:
    left, top, right, bottom = bbox
    if not (
        0 <= left < right <= image.width
        and 0 <= top < bottom <= image.height
    ):
        raise ValueError(
            f"Layout crop {bbox} is outside the {image.width}x{image.height} page"
        )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", Image.DecompressionBombWarning)
        return image.crop(bbox)


def _layout_retry_crop(
    image: Image.Image,
    bbox: tuple[int, int, int, int],
) -> Image.Image:
    """Add page context and sane geometry for an empty layout-crop retry."""
    left, top, right, bottom = bbox
    width = right - left
    height = bottom - top
    margin_x = max(12, ceil(width * 0.08))
    margin_y = max(12, ceil(height * 0.15))
    expanded_bbox = (
        max(0, left - margin_x),
        max(0, top - margin_y),
        min(image.width, right + margin_x),
        min(image.height, bottom + margin_y),
    )
    crop = _crop_pil_image(image, expanded_bbox).convert("RGB")
    corners = (
        crop.getpixel((0, 0)),
        crop.getpixel((crop.width - 1, 0)),
        crop.getpixel((0, crop.height - 1)),
        crop.getpixel((crop.width - 1, crop.height - 1)),
    )
    background = tuple(
        sum(int(color[channel]) for color in corners) // len(corners)
        for channel in range(3)
    )
    canvas_width = max(
        crop.width,
        ceil(crop.height / _LAYOUT_RETRY_MAX_ASPECT_RATIO),
    )
    canvas_height = max(
        crop.height,
        ceil(crop.width / _LAYOUT_RETRY_MAX_ASPECT_RATIO),
    )
    if (canvas_width, canvas_height) != crop.size:
        canvas = Image.new("RGB", (canvas_width, canvas_height), background)
        canvas.paste(
            crop,
            ((canvas_width - crop.width) // 2, (canvas_height - crop.height) // 2),
        )
        crop = canvas
    short_side = min(crop.size)
    if short_side < _LAYOUT_RETRY_MIN_SHORT_SIDE:
        scale = min(4.0, _LAYOUT_RETRY_MIN_SHORT_SIDE / short_side)
        crop = crop.resize(
            (ceil(crop.width * scale), ceil(crop.height * scale)),
            Image.Resampling.LANCZOS,
        )
    return crop


def _layout_task_type(label: str, declared: object) -> str:
    if declared in {"skip", "figure"}:
        return "figure"
    if isinstance(declared, str) and declared in _LAYOUT_LABEL_TASKS:
        return declared
    for task_type, labels in _LAYOUT_LABEL_TASKS.items():
        if label in labels:
            return "figure" if task_type == "skip" else task_type
    return "text"


def _detect_layout_regions(
    detector: Any,
    image: Image.Image,
) -> list[LayoutRegion]:
    """Run a started GLM-OCR PPDocLayoutDetector for one rendered page."""
    if not hasattr(detector, "process"):
        raise TypeError("layout_detector must provide process(images, ...)")
    output = detector.process([image], save_visualization=False)
    pages = output[0] if isinstance(output, tuple) else output
    if not isinstance(pages, list) or len(pages) != 1:
        raise ValueError("PP-DocLayout returned an unexpected page result")
    raw_regions = pages[0]
    if not isinstance(raw_regions, list):
        raise ValueError("PP-DocLayout returned malformed regions")

    sortable: list[tuple[int, int, dict[str, Any]]] = []
    for position, raw_region in enumerate(raw_regions):
        if not isinstance(raw_region, dict):
            raise ValueError("PP-DocLayout returned a malformed region")
        declared_order = raw_region.get("index", position)
        order = declared_order if isinstance(declared_order, int) else position
        sortable.append((order, position, raw_region))
    sortable.sort(key=lambda item: (item[0], item[1]))

    regions: list[LayoutRegion] = []
    for reading_order, (_, _, raw_region) in enumerate(sortable, start=1):
        label = str(raw_region.get("label", "text")).strip() or "text"
        score = float(raw_region.get("score", 0.0))
        if "bbox_2d" in raw_region:
            raw_bbox = raw_region["bbox_2d"]
            if not isinstance(raw_bbox, (list, tuple)) or len(raw_bbox) != 4:
                raise ValueError("PP-DocLayout returned an invalid bbox_2d")
            x1, y1, x2, y2 = (
                float(raw_bbox[0]) * image.width / 1000,
                float(raw_bbox[1]) * image.height / 1000,
                float(raw_bbox[2]) * image.width / 1000,
                float(raw_bbox[3]) * image.height / 1000,
            )
        else:
            raw_bbox = raw_region.get("coordinate", raw_region.get("bbox"))
            if not isinstance(raw_bbox, (list, tuple)) or len(raw_bbox) != 4:
                raise ValueError(
                    "PP-DocLayout returned a region without a bounding box"
                )
            x1, y1, x2, y2 = (float(value) for value in raw_bbox)
        bbox = (
            max(0, min(image.width, int(round(x1)))),
            max(0, min(image.height, int(round(y1)))),
            max(0, min(image.width, int(round(x2)))),
            max(0, min(image.height, int(round(y2)))),
        )
        if bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
            continue
        regions.append(
            LayoutRegion(
                reading_order,
                label,
                score,
                bbox,
                cast(
                    Literal["text", "table", "formula", "figure"],
                    _layout_task_type(label, raw_region.get("task_type")),
                ),
            )
        )
    if not regions:
        regions.append(
            LayoutRegion(1, "text", 0.0, (0, 0, image.width, image.height), "text")
        )
    return regions


def _data_url(image_bytes: bytes, mime_type: str) -> str:
    encoded = base64.b64encode(image_bytes).decode("ascii")
    return f"data:{mime_type};base64,{encoded}"


def _image_data_url(
    pixmap: pymupdf.Pixmap,
    max_data_url_bytes: int = _DEFAULT_MAX_DATA_URL_BYTES,
) -> tuple[str, str]:
    if max_data_url_bytes <= 0:
        raise ValueError("max_data_url_bytes must be greater than zero")
    png_url = _data_url(pixmap.tobytes("png"), "image/png")
    if len(png_url) <= max_data_url_bytes:
        return png_url, "image/png"

    jpeg_url = _data_url(
        pixmap.tobytes("jpeg", jpg_quality=92),
        "image/jpeg",
    )
    if len(jpeg_url) <= max_data_url_bytes:
        return jpeg_url, "image/jpeg"
    raise ValueError(
        "Image payload exceeds max_data_url_bytes after JPEG fallback; "
        "reduce PDF dpi or the source image dimensions"
    )

In [ ]:
# | export
_REF_RE = re.compile(r"<\|ref\|>(.*?)<\|/ref\|>", re.DOTALL)
_DET_RE = re.compile(r"<\|det\|>.*?<\|/det\|>", re.DOTALL)


def clean_grounding(raw: str) -> str:
    """Unwrap reference text and remove Unlimited-OCR coordinate markers."""
    cleaned = _REF_RE.sub(lambda match: match.group(1), raw)
    cleaned = _DET_RE.sub("", cleaned)
    lines = [line.rstrip() for line in cleaned.replace("\r\n", "\n").split("\n")]
    normalized: list[str] = []
    blank = False
    for line in lines:
        if line:
            normalized.append(line)
            blank = False
        elif normalized and not blank:
            normalized.append("")
            blank = True
    return "\n".join(normalized).strip()


def _layout_repetition_reason(content: str) -> str | None:
    fence_count = content.count("```")
    if fence_count >= 12:
        return f"contains {fence_count} Markdown fences"

    lines = [line.strip() for line in content.splitlines() if line.strip()]
    if lines:
        repeated_line, line_count = Counter(lines).most_common(1)[0]
        if line_count >= 8:
            return (
                f"repeats one {len(repeated_line)}-character line "
                f"{line_count} times"
            )

    compact = "".join(content.split())
    block_size = 32
    if len(compact) >= block_size * 2:
        blocks = Counter(
            compact[index : index + block_size]
            for index in range(len(compact) - block_size + 1)
        )
        _, block_count = blocks.most_common(1)[0]
        if block_count >= 8:
            return f"repeats a {block_size}-character block {block_count} times"
    return None


def _semantic_line_matches(left: str, right: str) -> bool:
    if left == right:
        return True
    shorter, longer = sorted((left, right), key=len)
    return (
        len(shorter) >= 4
        and len(shorter) / len(longer) >= 0.8
        and shorter in longer
    )


def _cut_at_repeated_character_block(content: str) -> str:
    block_size = 32
    compact_characters: list[str] = []
    source_indices: list[int] = []
    for source_index, character in enumerate(content):
        if not character.isspace():
            compact_characters.append(character)
            source_indices.append(source_index)
    compact = "".join(compact_characters)
    if len(compact) < block_size * 2:
        return content
    blocks = Counter(
        compact[index : index + block_size]
        for index in range(len(compact) - block_size + 1)
    )
    repeated_block, block_count = blocks.most_common(1)[0]
    if block_count < 8:
        return content
    first = compact.find(repeated_block)
    second = compact.find(repeated_block, first + 1)
    if second <= 0:
        return content
    return content[: source_indices[second]].rstrip()


def _recover_repeated_text(
    content: str,
    *,
    single_line: bool = False,
) -> str | None:
    semantic_lines = [
        line.strip()
        for line in content.splitlines()
        if line.strip() and not line.strip().startswith(("```", "~~~"))
    ]
    normalized_lines = [
        "".join(character.casefold() for character in line if character.isalnum())
        for line in semantic_lines
    ]
    counts = Counter(normalized_lines)
    selected_lines: list[str] = []
    selected_normalized: list[str] = []
    for line, normalized in zip(semantic_lines, normalized_lines, strict=True):
        matching_index = next(
            (
                index
                for index, previous in enumerate(selected_normalized)
                if _semantic_line_matches(normalized, previous)
            ),
            None,
        )
        if matching_index is not None:
            if len(normalized) > len(selected_normalized[matching_index]):
                selected_lines[matching_index] = line
            break
        if counts[normalized] >= 8:
            if not selected_lines:
                selected_lines.append(line)
            break
        selected_lines.append(line)
        selected_normalized.append(normalized)
    if single_line:
        selected_lines = selected_lines[:1]
    candidate = _cut_at_repeated_character_block(
        "\n".join(selected_lines).strip()
    )
    if single_line:
        candidate = candidate.lstrip("•●·- \t")
    if (
        not candidate
        or not any(character.isalnum() for character in candidate)
        or _layout_repetition_reason(candidate) is not None
    ):
        return None
    return candidate


def _recover_complete_html_table(content: str) -> str | None:
    casefolded = content.casefold()
    start = casefolded.find("<table")
    if start < 0:
        return None
    end = casefolded.find("</table>", start)
    if end < 0:
        return None
    return content[start : end + len("</table>")].strip() or None


def _validated_layout_content(
    content: str,
    *,
    finish_reason: str | None,
    task_type: str,
    label: str,
) -> tuple[str, str | None]:
    repetition_reason = (
        _layout_repetition_reason(content) if task_type == "text" else None
    )
    recovered: str | None = None
    if repetition_reason is not None:
        recovered = _recover_repeated_text(
            content,
            single_line=label
            in {
                "doc_title",
                "figure_title",
                "footer",
                "formula_number",
                "header",
                "number",
                "paragraph_title",
            },
        )
    elif task_type == "table" and finish_reason == "length":
        recovered = _recover_complete_html_table(content)
    if recovered is not None:
        reason = repetition_reason or "output continued after a complete HTML table"
        return recovered, (
            "Recovered the first complete non-repeating "
            f"{task_type} unit after Unlimited-OCR degeneration: {reason}"
        )
    if (
        repetition_reason is not None
        and task_type == "text"
        and label in {"footer", "header", "number"}
    ):
        return "", (
            "Discarded a degenerate Unlimited-OCR response for a decorative "
            f"or empty {label} region: {repetition_reason}"
        )
    if repetition_reason is not None:
        raise ValueError(
            "Unlimited-OCR returned a pathological repeated response: "
            f"{repetition_reason}"
        )
    if finish_reason == "length":
        raise ValueError(
            "Unlimited-OCR reached its output-token limit before completion"
        )
    return content, None


def _response_record(response: object) -> tuple[str, dict[str, Any]]:
    choices = _value(response, "choices")
    if not isinstance(choices, (list, tuple)) or not choices:
        raise ValueError("Unlimited-OCR returned no choices")
    choice = choices[0]
    message = _value(choice, "message")
    content = _value(message, "content")
    if not isinstance(content, str) or not content.strip():
        raise _EmptyOCROutputError("Unlimited-OCR returned an empty response")

    usage_obj = _value(response, "usage")
    usage: dict[str, int] = {}
    if usage_obj is not None:
        for name in ("prompt_tokens", "completion_tokens", "total_tokens"):
            count = _value(usage_obj, name)
            if isinstance(count, int):
                usage[name] = count
    return content, {
        "response_id": _value(response, "id"),
        "finish_reason": _value(choice, "finish_reason"),
        "usage": usage,
    }


def _vllm_extra_body(ngram_size: int, ngram_window: int) -> dict[str, Any]:
    if ngram_size <= 0:
        raise ValueError("ngram_size must be greater than zero")
    if ngram_window <= 0:
        raise ValueError("ngram_window must be greater than zero")
    return {
        "skip_special_tokens": False,
        "vllm_xargs": {
            "ngram_size": ngram_size,
            "window_size": ngram_window,
        },
    }


async def _parse_pixmap(
    pixmap: pymupdf.Pixmap,
    page_number: int,
    *,
    client: AsyncOpenAI,
    model: str,
    prompt: str,
    max_tokens: int,
    ngram_size: int,
    ngram_window: int,
    max_data_url_bytes: int,
    clean_output: bool,
    request_semaphore: asyncio.Semaphore | None,
) -> tuple[str, dict[str, Any]]:
    data_url, mime_type = _image_data_url(pixmap, max_data_url_bytes)
    started_at = perf_counter()

    async def send_request() -> object:
        return await client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url", "image_url": {"url": data_url}},
                    ],
                }
            ],
            max_tokens=max_tokens,
            temperature=0.0,
            extra_body=_vllm_extra_body(ngram_size, ngram_window),
        )

    if request_semaphore is None:
        response = await send_request()
    else:
        async with request_semaphore:
            response = await send_request()
    raw_content, response_metadata = _response_record(response)
    content = clean_grounding(raw_content) if clean_output else raw_content.strip()
    if not content:
        raise _EmptyOCROutputError(
            "Unlimited-OCR output is empty after post-processing"
        )
    return content, {
        "page_number": page_number,
        "width": pixmap.width,
        "height": pixmap.height,
        "input_mime_type": mime_type,
        "elapsed_s": perf_counter() - started_at,
        "raw_content": raw_content,
        "content": content,
        **response_metadata,
    }

In [ ]:
# | export
def _source_signature(source: Path) -> dict[str, Any]:
    stat = source.stat()
    return {
        "path": str(source),
        "size": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    }


def _publish_output_pair(
    markdown_path: Path,
    markdown: str,
    metadata: dict[str, Any],
) -> None:
    """Publish metadata first and Markdown last, restoring old outputs on error."""
    metadata_path = _metadata_path(markdown_path)
    markdown_path.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(
        dir=markdown_path.parent,
        prefix=f".{markdown_path.stem}.unlimited-",
    ) as temporary_directory:
        stage_root = Path(temporary_directory)
        stage_markdown = stage_root / markdown_path.name
        stage_metadata = stage_root / metadata_path.name
        stage_markdown.write_text(markdown, encoding="utf-8", newline="\n")
        stage_metadata.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
            newline="\n",
        )

        publications = [
            (stage_metadata, metadata_path),
            (stage_markdown, markdown_path),
        ]
        backups: list[tuple[Path, Path]] = []
        published: list[tuple[Path, Path]] = []
        try:
            for index, (_, final_path) in enumerate(publications):
                if final_path.exists():
                    backup_path = stage_root / f"backup-{index}"
                    final_path.replace(backup_path)
                    backups.append((backup_path, final_path))
            for staged_path, final_path in publications:
                staged_path.replace(final_path)
                published.append((final_path, staged_path))
        except Exception:
            for final_path, staged_path in reversed(published):
                if final_path.exists():
                    final_path.replace(staged_path)
            for backup_path, final_path in reversed(backups):
                if backup_path.exists():
                    backup_path.replace(final_path)
            raise


def _target_state(
    source: Path,
    target: Path,
    *,
    overwrite: bool,
    metadata_path: Path | None = None,
) -> UnlimitedOCRResult | None:
    selected_metadata_path = metadata_path or _metadata_path(target)
    if target.exists() and not target.is_file():
        return UnlimitedOCRResult(
            source,
            target,
            selected_metadata_path,
            "failed",
            error="Markdown target is not a file",
        )
    if selected_metadata_path.exists() and not selected_metadata_path.is_file():
        return UnlimitedOCRResult(
            source,
            target,
            selected_metadata_path,
            "failed",
            error="Metadata target is not a file",
        )
    if target.is_file() and not overwrite:
        return UnlimitedOCRResult(source, target, selected_metadata_path, "skipped")
    return None


def _assembled_markdown(page_contents: list[str]) -> str:
    sections = [
        f"<!-- Page {page_number} -->\n\n{content}"
        for page_number, content in enumerate(page_contents, start=1)
    ]
    return "\n\n".join(sections).rstrip() + "\n"


def _document_metadata(
    source: Path,
    *,
    service_base_url: str,
    model: str,
    prompt: str,
    dpi: int | None,
    max_page_pixels: int | None,
    max_tokens: int,
    ngram_size: int,
    ngram_window: int,
    clean_output: bool,
    elapsed_s: float,
    pages: list[dict[str, Any]],
) -> dict[str, Any]:
    return {
        "schema_version": 1,
        "provider": "baidu/Unlimited-OCR",
        "source": str(source),
        "source_signature": _source_signature(source),
        "service_base_url": service_base_url,
        "model": model,
        "prompt": prompt,
        "dpi": dpi,
        "max_page_pixels": max_page_pixels,
        "max_tokens": max_tokens,
        "decode_recipe": {
            "skip_special_tokens": False,
            "ngram_size": ngram_size,
            "ngram_window": ngram_window,
        },
        "clean_output": clean_output,
        "elapsed_s": elapsed_s,
        "pages": pages,
    }

In [ ]:
# | export
def _layout_asset_path(
    target: Path,
    page_number: int,
    region: LayoutRegion | None,
) -> Path:
    if region is None:
        filename = f"page-{page_number:04d}.png"
    else:
        label = (
            "".join(
                character if character.isalnum() else "-" for character in region.label
            ).strip("-")
            or "region"
        )
        filename = f"page-{page_number:04d}-region-{region.index:03d}-{label}.png"
    return Path(f"{target.stem}.assets") / filename


def _render_layout_region(
    page_number: int,
    record: dict[str, Any],
    *,
    asset_prefix: str = "",
) -> str:
    index = int(record["index"])
    label = str(record["label"])
    task_type = str(record["task_type"])
    status = str(record.get("status", "pending"))
    bbox = [int(value) for value in record["bbox"]]
    bbox_text = ",".join(str(value) for value in bbox)
    provenance = (
        f"layout-region page={page_number} index={index} label={label} "
        f"bbox={bbox_text} status={status}"
    )
    blocks: list[str] = []
    asset = record.get("asset")
    asset_link = f"{asset_prefix}{asset}" if isinstance(asset, str) else None
    if asset_link and task_type in {"figure", "table"}:
        kind = "Figure" if task_type == "figure" else "Table"
        alt = label.replace("_", " ").strip().title() or kind
        blocks.append(f"![{alt} {index}](<{asset_link}>)")
    elif asset_link and status == "failed":
        blocks.append(f"![Failed {label} region {index}](<{asset_link}>)")
    elif asset_link and status == "preserved":
        blocks.append(f"![Preserved {label} region {index}](<{asset_link}>)")

    if status == "pending":
        blocks.append(f"> [!NOTE] OCR pending for `{label}` region {index}.")
    elif status == "failed":
        error = str(record.get("error") or "unknown OCR failure").replace("\n", " ")
        blocks.append(
            f"> [!WARNING] OCR failed for `{label}` region {index}: {error}"
        )
    elif status == "preserved" and record.get("recovery"):
        blocks.append(
            f"> [!NOTE] {str(record['recovery']).replace(chr(10), ' ')}"
        )

    normalized = str(record.get("content") or "").strip()
    if normalized:
        if label == "doc_title" and not normalized.startswith("#"):
            normalized = f"# {normalized}"
        elif label == "paragraph_title" and not normalized.startswith("#"):
            normalized = f"## {normalized}"
        elif label == "list" and not normalized.startswith(("- ", "* ", "1. ")):
            normalized = "\n".join(
                f"- {line.strip()}"
                for line in normalized.splitlines()
                if line.strip()
            )
        elif task_type == "formula" and not normalized.startswith("$$"):
            normalized = f"$$\n{normalized}\n$$"
        blocks.append(normalized)

    return f"<!-- {provenance} -->\n\n" + (
        "\n\n".join(blocks) if blocks else f"<!-- Empty {label} region -->"
    )


def _render_layout_page(
    page_record: dict[str, Any] | None,
    page_number: int,
    *,
    asset_prefix: str = "",
) -> str:
    if page_record is None:
        return f"<!-- Page {page_number}: layout pending -->"
    sections: list[str] = []
    page_asset = page_record.get("page_asset")
    if isinstance(page_asset, str):
        sections.append(
            f"![Original page {page_number}](<{asset_prefix}{page_asset}>)"
        )
    sections.extend(
        _render_layout_region(page_number, record, asset_prefix=asset_prefix)
        for record in page_record.get("regions", [])
    )
    return f"<!-- Page {page_number} -->\n\n" + "\n\n".join(sections)


def _layout_counts(layout: dict[str, Any]) -> tuple[int, int, int]:
    regions = [
        region
        for page in layout.get("pages", [])
        if isinstance(page, dict)
        for region in page.get("regions", [])
    ]
    completed = sum(
        region.get("status") in {"completed", "preserved", "recovered"}
        for region in regions
    )
    failed = sum(region.get("status") == "failed" for region in regions)
    return len(regions), completed, failed


def _render_layout_document(
    layout: dict[str, Any],
    *,
    partial: bool,
    asset_prefix: str = "",
) -> str:
    pages = layout.get("pages", [])
    sections: list[str] = []
    if partial:
        pages_completed = sum(
            isinstance(page, dict) and page.get("status") in {"completed", "partial"}
            for page in pages
        )
        regions_total, regions_completed, regions_failed = _layout_counts(layout)
        sections.append(
            "> [!NOTE] OCR processing is incomplete. "
            f"Pages: {pages_completed}/{layout['pages_total']}; "
            f"regions: {regions_completed}/{regions_total} complete, "
            f"{regions_failed} failed."
        )
    sections.extend(
        _render_layout_page(page, page_number, asset_prefix=asset_prefix)
        for page_number, page in enumerate(pages, start=1)
    )
    return "\n\n".join(sections).rstrip() + "\n"


def _layout_region_record(
    region: LayoutRegion,
    page_number: int,
    target: Path,
    *,
    prompt: str,
    max_tokens: int,
) -> dict[str, Any]:
    asset_path = (
        _layout_asset_path(target, page_number, region)
        if region.task_type in _LAYOUT_ASSET_TASKS
        else None
    )
    region_max_tokens = (
        min(max_tokens, _DEFAULT_LAYOUT_TEXT_MAX_TOKENS)
        if region.task_type == "text"
        else max_tokens
    )
    return {
        "index": region.index,
        "label": region.label,
        "score": region.score,
        "bbox": list(region.bbox),
        "task_type": region.task_type,
        "status": "preserved" if region.task_type == "figure" else "pending",
        "prompt": None if region.task_type == "figure" else prompt,
        "max_tokens": (
            None if region.task_type == "figure" else region_max_tokens
        ),
        "asset": asset_path.as_posix() if asset_path is not None else None,
        "raw_content": "",
        "content": "",
        "error": None,
        "recovery": None,
    }


def _regions_from_page_record(page_record: dict[str, Any]) -> list[LayoutRegion]:
    return [
        LayoutRegion(
            int(record["index"]),
            str(record["label"]),
            float(record.get("score", 0.0)),
            cast(tuple[int, int, int, int], tuple(int(v) for v in record["bbox"])),
            cast(
                Literal["text", "table", "formula", "figure"],
                str(record["task_type"]),
            ),
        )
        for record in page_record.get("regions", [])
    ]


def _save_region_asset(
    crop: pymupdf.Pixmap,
    record: dict[str, Any],
    *,
    target: Path,
    page_number: int,
    region: LayoutRegion,
    stage_assets: Path,
) -> None:
    if record.get("asset") is None:
        record["asset"] = _layout_asset_path(
            target,
            page_number,
            region,
        ).as_posix()
    asset_file = stage_assets.parent / str(record["asset"])
    asset_file.parent.mkdir(parents=True, exist_ok=True)
    if not asset_file.is_file():
        asset_file.write_bytes(crop.tobytes("png"))


def _prepare_layout_page(
    pixmap: pymupdf.Pixmap,
    page_number: int,
    *,
    target: Path,
    stage_assets: Path,
    detector: Any,
    prompt: str,
    embed_page_image: bool,
    max_tokens: int,
    existing_record: dict[str, Any] | None,
    requested_dpi: int | None = None,
    effective_dpi: int | None = None,
    detected_regions: list[LayoutRegion] | None = None,
) -> tuple[Image.Image, list[LayoutRegion], dict[str, Any]]:
    image = _pixmap_to_pil(pixmap)
    if existing_record is None:
        regions = detected_regions or _detect_layout_regions(detector, image)
        page_record: dict[str, Any] = {
            "page_number": page_number,
            "width": image.width,
            "height": image.height,
            "render_pixels": image.width * image.height,
            "requested_dpi": requested_dpi,
            "effective_dpi": effective_dpi,
            "status": "processing",
            "error": None,
            "regions": [
                _layout_region_record(
                    region,
                    page_number,
                    target,
                    prompt=prompt,
                    max_tokens=max_tokens,
                )
                for region in regions
            ],
        }
    else:
        page_record = copy.deepcopy(existing_record)
        page_record["status"] = "processing"
        page_record["width"] = image.width
        page_record["height"] = image.height
        page_record["render_pixels"] = image.width * image.height
        page_record["requested_dpi"] = requested_dpi
        page_record["effective_dpi"] = effective_dpi
        regions = _regions_from_page_record(page_record)

    if embed_page_image:
        page_asset = _layout_asset_path(target, page_number, None)
        page_file = stage_assets.parent / page_asset
        page_file.parent.mkdir(parents=True, exist_ok=True)
        if not page_file.is_file():
            page_file.write_bytes(pixmap.tobytes("png"))
        page_record["page_asset"] = page_asset.as_posix()

    records_by_index = {
        int(record["index"]): record for record in page_record["regions"]
    }
    for region in regions:
        record = records_by_index[region.index]
        if region.task_type in _LAYOUT_ASSET_TASKS:
            _save_region_asset(
                _pil_to_pixmap(_crop_pil_image(image, region.bbox)),
                record,
                target=target,
                page_number=page_number,
                region=region,
                stage_assets=stage_assets,
            )
    return image, regions, page_record


async def _layout_page_async(
    pixmap: pymupdf.Pixmap,
    page_number: int,
    *,
    target: Path,
    stage_assets: Path,
    detector: Any,
    client: AsyncOpenAI,
    model: str,
    prompt: str,
    embed_page_image: bool,
    request_semaphore: asyncio.Semaphore | None,
    layout_semaphore: asyncio.Semaphore | None,
    max_tokens: int,
    ngram_size: int,
    ngram_window: int,
    max_data_url_bytes: int,
    clean_output: bool,
    requested_dpi: int | None = None,
    effective_dpi: int | None = None,
    existing_record: dict[str, Any] | None = None,
    checkpoint: Callable[[dict[str, Any]], Awaitable[None]] | None = None,
) -> tuple[str, dict[str, Any]]:
    image = _pixmap_to_pil(pixmap)
    if existing_record is not None:
        regions = _regions_from_page_record(existing_record)
    elif layout_semaphore is None:
        regions = await asyncio.to_thread(_detect_layout_regions, detector, image)
    else:
        async with layout_semaphore:
            regions = await asyncio.to_thread(_detect_layout_regions, detector, image)
    image, regions, page_record = _prepare_layout_page(
        pixmap,
        page_number,
        target=target,
        stage_assets=stage_assets,
        detector=detector,
        prompt=prompt,
        embed_page_image=embed_page_image,
        max_tokens=max_tokens,
        existing_record=existing_record,
        requested_dpi=requested_dpi,
        effective_dpi=effective_dpi,
        detected_regions=regions,
    )
    if checkpoint is not None:
        await checkpoint(page_record)
    records_by_index = {
        int(record["index"]): record for record in page_record["regions"]
    }

    async def recognize(
        region: LayoutRegion,
    ) -> tuple[int, str, dict[str, Any] | str, str | None]:
        record = records_by_index[region.index]
        crop = _pil_to_pixmap(_crop_pil_image(image, region.bbox))
        recovery_notes: list[str] = []

        async def parse_region(candidate: pymupdf.Pixmap) -> tuple[str, dict[str, Any]]:
            return await _parse_pixmap(
                candidate,
                page_number,
                client=client,
                model=model,
                prompt=str(record["prompt"]),
                max_tokens=int(record["max_tokens"]),
                ngram_size=ngram_size,
                ngram_window=ngram_window,
                max_data_url_bytes=max_data_url_bytes,
                clean_output=clean_output,
                request_semaphore=request_semaphore,
            )

        try:
            retrying_legacy_empty = _is_empty_ocr_error(record.get("error"))
            candidate = (
                _pil_to_pixmap(_layout_retry_crop(image, region.bbox))
                if retrying_legacy_empty
                else crop
            )
            if retrying_legacy_empty:
                recovery_notes.append(
                    "Retried a previous empty OCR result with expanded, "
                    "aspect-ratio-padded page context."
                )
            try:
                content, response_record = await parse_region(candidate)
            except _EmptyOCROutputError:
                if retrying_legacy_empty:
                    raise
                candidate = _pil_to_pixmap(_layout_retry_crop(image, region.bbox))
                recovery_notes.append(
                    "Retried an empty OCR result with expanded, "
                    "aspect-ratio-padded page context."
                )
                content, response_record = await parse_region(candidate)
            validated, validation_recovery = _validated_layout_content(
                content,
                finish_reason=cast(
                    str | None,
                    response_record.get("finish_reason"),
                ),
                task_type=region.task_type,
                label=region.label,
            )
            if validation_recovery is not None:
                recovery_notes.append(validation_recovery)
            recovery = " ".join(recovery_notes) or None
            if recovery is not None:
                _save_region_asset(
                    crop,
                    record,
                    target=target,
                    page_number=page_number,
                    region=region,
                    stage_assets=stage_assets,
                )
            response_record["content"] = validated
            return (
                region.index,
                "recovered" if recovery is not None else "completed",
                response_record,
                recovery,
            )
        except _EmptyOCROutputError:
            _save_region_asset(
                crop,
                record,
                target=target,
                page_number=page_number,
                region=region,
                stage_assets=stage_assets,
            )
            preservation = (
                "Unlimited-OCR returned no usable text after a contextual retry; "
                "the original region image was preserved instead."
            )
            return region.index, "preserved", "", preservation
        except Exception as error:
            _save_region_asset(
                crop,
                record,
                target=target,
                page_number=page_number,
                region=region,
                stage_assets=stage_assets,
            )
            return (
                region.index,
                "failed",
                f"{type(error).__name__}: {error}",
                None,
            )

    tasks = [
        asyncio.create_task(recognize(region))
        for region in regions
        if records_by_index[region.index]["status"]
        not in {"completed", "preserved", "recovered"}
    ]
    try:
        for completed_task in asyncio.as_completed(tasks):
            region_index, status, value, recovery = await completed_task
            record = records_by_index[region_index]
            if status in {"completed", "recovered"}:
                assert isinstance(value, dict)
                record.update(
                    status=status,
                    raw_content=str(value.get("raw_content") or ""),
                    content=str(value.get("content") or ""),
                    input_mime_type=value.get("input_mime_type"),
                    elapsed_s=value.get("elapsed_s"),
                    response_id=value.get("response_id"),
                    finish_reason=value.get("finish_reason"),
                    usage=value.get("usage", {}),
                    error=None,
                    recovery=recovery,
                )
            elif status == "preserved":
                record.update(
                    status="preserved",
                    raw_content="",
                    content="",
                    error=None,
                    recovery=recovery,
                )
            else:
                record.update(
                    status="failed",
                    raw_content="",
                    content="",
                    error=value,
                    recovery=None,
                )
            if checkpoint is not None:
                await checkpoint(page_record)
    finally:
        for task in tasks:
            if not task.done():
                task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)
    page_record["status"] = (
        "partial"
        if any(record["status"] == "failed" for record in page_record["regions"])
        else "completed"
    )
    if checkpoint is not None:
        await checkpoint(page_record)
    return _render_layout_page(page_record, page_number), page_record


def _publish_layout_bundle(
    stage_root: Path,
    target: Path,
    markdown: str,
    layout: dict[str, Any],
) -> None:
    """Publish assets and sidecar first, then atomically expose Markdown last."""
    stage_markdown = stage_root / target.name
    stage_sidecar = stage_root / f"{target.stem}.layout.json"
    stage_assets = stage_root / f"{target.stem}.assets"
    stage_assets.mkdir(parents=True, exist_ok=True)
    stage_markdown.write_text(markdown, encoding="utf-8", newline="\n")
    stage_sidecar.write_text(
        json.dumps(layout, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
        newline="\n",
    )

    final_sidecar = target.with_suffix(".layout.json")
    final_assets = target.parent / stage_assets.name
    publications = [
        (stage_assets, final_assets),
        (stage_sidecar, final_sidecar),
        (stage_markdown, target),
    ]
    backups: list[tuple[Path, Path]] = []
    published: list[tuple[Path, Path]] = []
    try:
        for backup_index, (_, final_path) in enumerate(publications):
            if final_path.exists():
                backup_path = stage_root / f"backup-{backup_index}"
                final_path.replace(backup_path)
                backups.append((backup_path, final_path))
        for staged_path, final_path in publications:
            staged_path.replace(final_path)
            published.append((final_path, staged_path))
    except Exception:
        for final_path, staged_path in reversed(published):
            if final_path.exists():
                final_path.replace(staged_path)
        for backup_path, final_path in reversed(backups):
            if backup_path.exists():
                backup_path.replace(final_path)
        raise


def _layout_workspace_paths(target: Path) -> tuple[Path, Path, Path, Path, Path]:
    work_root = target.parent / f".{target.stem}.layout-work"
    assets = work_root / f"{target.stem}.assets"
    checkpoint = work_root / "checkpoint.json"
    partial_markdown = target.with_name(f"{target.stem}.partial.md")
    partial_sidecar = target.with_name(f"{target.stem}.partial.layout.json")
    return work_root, assets, checkpoint, partial_markdown, partial_sidecar


def _layout_output_is_partial(sidecar: Path) -> bool:
    if not sidecar.is_file():
        return False
    try:
        return json.loads(sidecar.read_text(encoding="utf-8")).get("status") == "partial"
    except (OSError, ValueError, TypeError):
        return False


def _migrate_layout_settings(
    layout: dict[str, Any],
    settings: dict[str, Any],
    assets: Path,
) -> bool:
    """Migrate the adaptive-DPI addition without redoing compatible pages."""
    checkpoint_settings = layout.get("settings")
    if checkpoint_settings == settings:
        return False
    legacy_settings = dict(settings)
    legacy_settings.pop("max_page_pixels", None)
    can_migrate_adaptive_dpi = (
        isinstance(checkpoint_settings, dict)
        and "max_page_pixels" in settings
        and checkpoint_settings == legacy_settings
    )
    if not can_migrate_adaptive_dpi:
        raise ValueError("layout settings are incompatible")

    layout["settings"] = copy.deepcopy(settings)
    layout["max_page_pixels"] = settings["max_page_pixels"]
    max_page_pixels = int(settings["max_page_pixels"])
    for page_index, page_record in enumerate(layout["pages"], start=1):
        if not isinstance(page_record, dict):
            continue
        if page_record.get("status") == "completed":
            continue
        render_pixels = page_record.get("render_pixels")
        if isinstance(render_pixels, int) and render_pixels <= max_page_pixels:
            continue
        layout["pages"][page_index - 1] = None
        for asset_path in assets.glob(f"page-{page_index:04d}*"):
            if asset_path.is_dir() and not asset_path.is_symlink():
                shutil.rmtree(asset_path)
            else:
                asset_path.unlink(missing_ok=True)
    return True


def _prepare_layout_workspace(
    source: Path,
    target: Path,
    *,
    pages_total: int,
    settings: dict[str, Any],
    resume_partial: bool,
    overwrite: bool,
) -> tuple[Path, Path, dict[str, Any]]:
    work_root, assets, checkpoint, partial_markdown, partial_sidecar = (
        _layout_workspace_paths(target)
    )
    expected_signature = _source_signature(source)
    if work_root.exists() and overwrite:
        shutil.rmtree(work_root)
        partial_markdown.unlink(missing_ok=True)
        partial_sidecar.unlink(missing_ok=True)
    if work_root.exists():
        if not resume_partial:
            raise ValueError(
                f"Partial OCR checkpoint exists for {source}; enable "
                "resume_partial or use overwrite=True"
            )
        try:
            layout = json.loads(checkpoint.read_text(encoding="utf-8"))
        except Exception as error:
            raise ValueError(
                f"Cannot read partial OCR checkpoint {checkpoint}: {error}; "
                "use overwrite=True to start over"
            ) from error
        if (
            layout.get("source_signature") != expected_signature
            or layout.get("pages_total") != pages_total
        ):
            raise ValueError(
                f"Partial OCR checkpoint for {source} is incompatible with the "
                "current source or settings; use overwrite=True to start over"
            )
        assets.mkdir(parents=True, exist_ok=True)
        try:
            migrated = _migrate_layout_settings(layout, settings, assets)
        except ValueError as error:
            raise ValueError(
                f"Partial OCR checkpoint for {source} is incompatible with the "
                "current source or settings; use overwrite=True to start over"
            ) from error
        if migrated:
            _write_layout_checkpoint(target, layout)
        return work_root, assets, layout

    final_sidecar = target.with_suffix(".layout.json")
    if (
        resume_partial
        and target.is_file()
        and _layout_output_is_partial(final_sidecar)
    ):
        try:
            layout = json.loads(final_sidecar.read_text(encoding="utf-8"))
        except Exception as error:
            raise ValueError(
                f"Cannot restore partial OCR output {final_sidecar}: {error}; "
                "use overwrite=True to start over"
            ) from error
        if (
            layout.get("source_signature") != expected_signature
            or layout.get("pages_total") != pages_total
        ):
            raise ValueError(
                f"Partial OCR output for {source} is incompatible with the "
                "current source or settings; use overwrite=True to start over"
            )
        final_assets = target.parent / f"{target.stem}.assets"
        if final_assets.is_dir():
            _clone_layout_assets(final_assets, assets)
        else:
            assets.mkdir(parents=True)
        try:
            _migrate_layout_settings(layout, settings, assets)
        except ValueError as error:
            shutil.rmtree(work_root)
            raise ValueError(
                f"Partial OCR output for {source} is incompatible with the "
                "current source or settings; use overwrite=True to start over"
            ) from error
        layout["status"] = "processing"
        _write_layout_checkpoint(target, layout)
        return work_root, assets, layout

    assets.mkdir(parents=True)
    layout = {
        "schema_version": _LAYOUT_CHECKPOINT_SCHEMA_VERSION,
        "status": "processing",
        "source": str(source),
        "source_signature": expected_signature,
        "settings": settings,
        "provider": settings["provider"],
        "service_base_url": settings["service_base_url"],
        "model": settings["model"],
        "layout_model": settings["layout_model"],
        "dpi": settings["dpi"],
        "max_page_pixels": settings.get("max_page_pixels"),
        "pages_total": pages_total,
        "pages": [None] * pages_total,
    }
    _write_layout_checkpoint(target, layout)
    return work_root, assets, layout


def _write_layout_checkpoint(target: Path, layout: dict[str, Any]) -> None:
    work_root, _, checkpoint, partial_markdown, partial_sidecar = (
        _layout_workspace_paths(target)
    )
    asset_prefix = f"{work_root.name}/"
    snapshot = copy.deepcopy(layout)
    snapshot["status"] = "processing"
    _atomic_write_json(checkpoint, snapshot)
    partial_snapshot = copy.deepcopy(snapshot)
    partial_snapshot["asset_prefix"] = asset_prefix
    _atomic_write_json(partial_sidecar, partial_snapshot)
    _atomic_write_text(
        partial_markdown,
        _render_layout_document(snapshot, partial=True, asset_prefix=asset_prefix),
    )


def _clone_layout_assets(source: Path, destination: Path) -> None:
    try:
        shutil.copytree(source, destination, copy_function=os.link)
    except OSError:
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(source, destination, copy_function=shutil.copy2)


def _publish_completed_layout(
    target: Path,
    layout: dict[str, Any],
) -> tuple[str, int, int, int]:
    work_root, assets, _, partial_markdown, partial_sidecar = _layout_workspace_paths(
        target
    )
    regions_total, regions_completed, regions_failed = _layout_counts(layout)
    status = "partial" if regions_failed else "processed"
    final_layout = copy.deepcopy(layout)
    final_layout["status"] = status
    markdown = _render_layout_document(final_layout, partial=False)
    with TemporaryDirectory(
        dir=target.parent,
        prefix=f".{target.stem}.layout-publish-",
    ) as temporary_directory:
        stage_root = Path(temporary_directory)
        _clone_layout_assets(assets, stage_root / assets.name)
        _publish_layout_bundle(stage_root, target, markdown, final_layout)
    if status == "processed":
        shutil.rmtree(work_root)
        partial_markdown.unlink(missing_ok=True)
        partial_sidecar.unlink(missing_ok=True)
    else:
        _write_layout_checkpoint(target, layout)
    return status, regions_total, regions_completed, regions_failed


def _layout_settings(
    *,
    service_base_url: str,
    model: str,
    prompt: str,
    dpi: int | None,
    layout_model: str,
    layout_threshold: float,
    embed_page_image: bool,
    max_tokens: int,
    ngram_size: int,
    ngram_window: int,
    max_data_url_bytes: int,
    clean_output: bool,
    max_page_pixels: int | None = None,
) -> dict[str, Any]:
    settings = {
        "provider": "baidu/Unlimited-OCR",
        "service_base_url": service_base_url,
        "model": model,
        "prompt": prompt,
        "dpi": dpi,
        "layout_model": layout_model,
        "layout_threshold": layout_threshold,
        "embed_page_image": embed_page_image,
        "max_tokens": max_tokens,
        "ngram_size": ngram_size,
        "ngram_window": ngram_window,
        "max_data_url_bytes": max_data_url_bytes,
        "clean_output": clean_output,
        "response_validation_version": _LAYOUT_RESPONSE_VALIDATION_VERSION,
    }
    if max_page_pixels is not None:
        settings["max_page_pixels"] = max_page_pixels
    return settings


@contextmanager
def _exclusive_output_lock(lock_path: Path, description: str) -> Iterator[None]:
    lock_path.parent.mkdir(parents=True, exist_ok=True)
    with lock_path.open("a+", encoding="utf-8") as lock_file:
        try:
            fcntl.flock(lock_file.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            lock_file.seek(0)
            owner = lock_file.read().strip() or "unknown process"
            raise RuntimeError(
                f"Another OCR process holds the {description} lock: {owner}"
            ) from error
        lock_file.seek(0)
        lock_file.truncate()
        lock_file.write(
            json.dumps({"pid": os.getpid(), "description": description}) + "\n"
        )
        lock_file.flush()
        try:
            yield
        finally:
            fcntl.flock(lock_file.fileno(), fcntl.LOCK_UN)


def _atomic_write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path: Path | None = None
    try:
        with NamedTemporaryFile(
            mode="w",
            encoding="utf-8",
            newline="\n",
            dir=path.parent,
            prefix=f".{path.name}.",
            suffix=".tmp",
            delete=False,
        ) as temporary_file:
            temporary_file.write(content)
            temporary_path = Path(temporary_file.name)
        temporary_path.replace(path)
        temporary_path = None
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


def _atomic_write_json(path: Path, content: dict[str, Any]) -> None:
    _atomic_write_text(
        path,
        json.dumps(content, ensure_ascii=False, indent=2) + "\n",
    )

In [ ]:
# | export
async def _ocr_pdf_plain(
    pdf_path: Path | str,
    markdown_path: Path | str,
    *,
    client: AsyncOpenAI,
    service_base_url: str = _DEFAULT_BASE_URL,
    model: str | None = None,
    prompt: str = _DEFAULT_PROMPT,
    dpi: int = 300,
    max_page_pixels: int = _DEFAULT_MAX_PAGE_PIXELS,
    overwrite: bool = False,
    page_concurrency: int = 2,
    max_tokens: int = _DEFAULT_MAX_TOKENS,
    ngram_size: int = _DEFAULT_NGRAM_SIZE,
    ngram_window: int = _DEFAULT_NGRAM_WINDOW,
    max_data_url_bytes: int = _DEFAULT_MAX_DATA_URL_BYTES,
    clean_output: bool = True,
    request_semaphore: asyncio.Semaphore | None = None,
    page_started: Callable[[int], None] | None = None,
    page_progress: Callable[[int, int, float], None] | None = None,
) -> UnlimitedOCRResult:
    """Parse one PDF into page-ordered Unlimited-OCR Markdown."""
    source = Path(pdf_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    metadata_path = _metadata_path(target)
    selected_base_url = _resolve_base_url(service_base_url)
    selected_model = _resolve_model(model)
    selected_prompt = _resolve_prompt(prompt)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if max_page_pixels <= 0:
        raise ValueError("max_page_pixels must be greater than zero")
    if page_concurrency <= 0:
        raise ValueError("page_concurrency must be greater than zero")
    if max_tokens <= 0:
        raise ValueError("max_tokens must be greater than zero")
    _vllm_extra_body(ngram_size, ngram_window)
    if state := _target_state(source, target, overwrite=overwrite):
        return state

    started_at = perf_counter()
    pages_total = 0
    pages_completed = 0
    tasks: list[asyncio.Task[tuple[int, str, dict[str, Any]]]] = []
    try:
        if not source.is_file():
            raise FileNotFoundError(f"PDF does not exist: {source}")
        page_gate = asyncio.Semaphore(page_concurrency)
        with pymupdf.open(source) as document:
            if document.needs_pass:
                raise ValueError("PDF requires a password")
            pages_total = document.page_count
            if pages_total == 0:
                raise ValueError("PDF contains no pages")
            if page_started is not None:
                page_started(pages_total)

            async def process_page(
                page_number: int,
            ) -> tuple[int, str, dict[str, Any]]:
                nonlocal pages_completed
                page_started_at = perf_counter()
                async with page_gate:
                    page, load_warnings = _load_pdf_page(
                        document,
                        page_number - 1,
                    )
                    pixmap, effective_dpi, render_warnings = _render_pdf_page(
                        page,
                        dpi=dpi,
                        max_page_pixels=max_page_pixels,
                    )
                    render_warnings = tuple(
                        dict.fromkeys((*load_warnings, *render_warnings))
                    )
                    content, record = await _parse_pixmap(
                        pixmap,
                        page_number,
                        client=client,
                        model=selected_model,
                        prompt=selected_prompt,
                        max_tokens=max_tokens,
                        ngram_size=ngram_size,
                        ngram_window=ngram_window,
                        max_data_url_bytes=max_data_url_bytes,
                        clean_output=clean_output,
                        request_semaphore=request_semaphore,
                    )
                elapsed_s = perf_counter() - page_started_at
                record["elapsed_s"] = elapsed_s
                record["requested_dpi"] = dpi
                record["effective_dpi"] = effective_dpi
                record["render_width"] = pixmap.width
                record["render_height"] = pixmap.height
                record["render_pixels"] = pixmap.width * pixmap.height
                record["render_warnings"] = list(render_warnings)
                pages_completed += 1
                if page_progress is not None:
                    page_progress(pages_completed, pages_total, elapsed_s)
                return page_number, content, record

            tasks = [
                asyncio.create_task(process_page(page_number))
                for page_number in range(1, pages_total + 1)
            ]
            try:
                parsed_pages = await asyncio.gather(*tasks)
            finally:
                for task in tasks:
                    if not task.done():
                        task.cancel()
                await asyncio.gather(*tasks, return_exceptions=True)

        parsed_pages.sort(key=lambda item: item[0])
        document_elapsed_s = perf_counter() - started_at
        _publish_output_pair(
            target,
            _assembled_markdown([content for _, content, _ in parsed_pages]),
            _document_metadata(
                source,
                service_base_url=selected_base_url,
                model=selected_model,
                prompt=selected_prompt,
                dpi=dpi,
                max_page_pixels=max_page_pixels,
                max_tokens=max_tokens,
                ngram_size=ngram_size,
                ngram_window=ngram_window,
                clean_output=clean_output,
                elapsed_s=document_elapsed_s,
                pages=[record for _, _, record in parsed_pages],
            ),
        )
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            "processed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
        )
    except Exception as error:
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            "failed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
            f"{type(error).__name__}: {error}",
        )

In [ ]:
# | export
async def _ocr_image_plain(
    image_path: Path | str,
    markdown_path: Path | str,
    *,
    client: AsyncOpenAI,
    service_base_url: str = _DEFAULT_BASE_URL,
    model: str | None = None,
    prompt: str = _DEFAULT_PROMPT,
    overwrite: bool = False,
    max_tokens: int = _DEFAULT_MAX_TOKENS,
    ngram_size: int = _DEFAULT_NGRAM_SIZE,
    ngram_window: int = _DEFAULT_NGRAM_WINDOW,
    max_data_url_bytes: int = _DEFAULT_MAX_DATA_URL_BYTES,
    clean_output: bool = True,
    request_semaphore: asyncio.Semaphore | None = None,
) -> UnlimitedOCRResult:
    """Parse one image into Unlimited-OCR Markdown at native resolution."""
    source = Path(image_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    metadata_path = _metadata_path(target)
    selected_base_url = _resolve_base_url(service_base_url)
    selected_model = _resolve_model(model)
    selected_prompt = _resolve_prompt(prompt)
    if max_tokens <= 0:
        raise ValueError("max_tokens must be greater than zero")
    _vllm_extra_body(ngram_size, ngram_window)
    if state := _target_state(source, target, overwrite=overwrite):
        return state

    started_at = perf_counter()
    pages_total = 0
    pages_completed = 0
    try:
        if not source.is_file():
            raise FileNotFoundError(f"Image does not exist: {source}")
        if source.suffix.casefold() not in _IMAGE_SUFFIXES:
            raise ValueError(f"Unsupported image type: {source.suffix or '<none>'}")
        pixmap = _load_image_pixmap(source)
        pages_total = 1
        content, page_record = await _parse_pixmap(
            pixmap,
            1,
            client=client,
            model=selected_model,
            prompt=selected_prompt,
            max_tokens=max_tokens,
            ngram_size=ngram_size,
            ngram_window=ngram_window,
            max_data_url_bytes=max_data_url_bytes,
            clean_output=clean_output,
            request_semaphore=request_semaphore,
        )
        pages_completed = 1
        document_elapsed_s = perf_counter() - started_at
        _publish_output_pair(
            target,
            _assembled_markdown([content]),
            _document_metadata(
                source,
                service_base_url=selected_base_url,
                model=selected_model,
                prompt=selected_prompt,
                dpi=None,
                max_page_pixels=None,
                max_tokens=max_tokens,
                ngram_size=ngram_size,
                ngram_window=ngram_window,
                clean_output=clean_output,
                elapsed_s=document_elapsed_s,
                pages=[page_record],
            ),
        )
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            "processed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
        )
    except Exception as error:
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            "failed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
            f"{type(error).__name__}: {error}",
        )

In [ ]:
# | export
async def _ocr_pdf_layout(
    pdf_path: Path | str,
    markdown_path: Path | str,
    *,
    client: AsyncOpenAI,
    service_base_url: str,
    model: str | None,
    prompt: str,
    dpi: int,
    max_page_pixels: int,
    overwrite: bool,
    page_concurrency: int,
    max_tokens: int,
    ngram_size: int,
    ngram_window: int,
    max_data_url_bytes: int,
    clean_output: bool,
    request_semaphore: asyncio.Semaphore | None,
    layout_detector: Any | None,
    layout_model: str,
    layout_threshold: float,
    layout_device: str | None,
    embed_page_image: bool,
    resume_partial: bool,
    layout_semaphore: asyncio.Semaphore | None,
    page_started: Callable[[int], None] | None,
    page_progress: Callable[[int, int, float], None] | None,
) -> UnlimitedOCRResult:
    source = Path(pdf_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    metadata_path = target.with_suffix(".layout.json")
    selected_base_url = _resolve_base_url(service_base_url)
    selected_model = _resolve_model(model)
    selected_prompt = _resolve_prompt(prompt)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if max_page_pixels <= 0:
        raise ValueError("max_page_pixels must be greater than zero")
    if page_concurrency <= 0:
        raise ValueError("page_concurrency must be greater than zero")
    if max_tokens <= 0:
        raise ValueError("max_tokens must be greater than zero")
    if not 0 <= layout_threshold <= 1:
        raise ValueError("layout_threshold must be between 0 and 1")
    if not layout_model.strip():
        raise ValueError("layout_model must not be empty")
    _vllm_extra_body(ngram_size, ngram_window)
    if state := _target_state(
        source,
        target,
        overwrite=overwrite,
        metadata_path=metadata_path,
    ):
        if not (
            state.status == "skipped"
            and resume_partial
            and _layout_output_is_partial(metadata_path)
        ):
            return state

    started_at = perf_counter()
    pages_total = 0
    pages_completed = 0
    owned_detector: Any | None = None
    try:
        if not source.is_file():
            raise FileNotFoundError(f"PDF does not exist: {source}")
        detector = layout_detector
        if detector is None:
            detector = await asyncio.to_thread(
                create_pp_doclayout_detector,
                model_name=layout_model,
                threshold=layout_threshold,
                device=layout_device,
            )
            owned_detector = detector
        target.parent.mkdir(parents=True, exist_ok=True)

        with pymupdf.open(source) as document:
            if document.needs_pass:
                raise ValueError("PDF requires a password")
            pages_total = document.page_count
            if pages_total == 0:
                raise ValueError("PDF contains no pages")
            if page_started is not None:
                page_started(pages_total)

            settings = _layout_settings(
                service_base_url=selected_base_url,
                model=selected_model,
                prompt=selected_prompt,
                dpi=dpi,
                max_page_pixels=max_page_pixels,
                layout_model=layout_model,
                layout_threshold=layout_threshold,
                embed_page_image=embed_page_image,
                max_tokens=max_tokens,
                ngram_size=ngram_size,
                ngram_window=ngram_window,
                max_data_url_bytes=max_data_url_bytes,
                clean_output=clean_output,
            )
            _, stage_assets, layout = _prepare_layout_workspace(
                source,
                target,
                pages_total=pages_total,
                settings=settings,
                resume_partial=resume_partial,
                overwrite=overwrite,
            )
            checkpoint_lock = asyncio.Lock()

            async def checkpoint_page(page_record: dict[str, Any]) -> None:
                async with checkpoint_lock:
                    layout["pages"][int(page_record["page_number"]) - 1] = (
                        copy.deepcopy(page_record)
                    )
                    snapshot = copy.deepcopy(layout)
                    await asyncio.to_thread(
                        _write_layout_checkpoint,
                        target,
                        snapshot,
                    )

            page_gate = asyncio.Semaphore(page_concurrency)
            page_records: list[dict[str, Any] | None] = list(layout["pages"])

            async def process_page(
                page_number: int,
            ) -> tuple[int, dict[str, Any], float]:
                nonlocal pages_completed
                page_started_at = perf_counter()
                existing_record = page_records[page_number - 1]
                if (
                    isinstance(existing_record, dict)
                    and existing_record.get("status") == "completed"
                ):
                    page_record = existing_record
                else:
                    async with page_gate:
                        page, load_warnings = _load_pdf_page(
                            document,
                            page_number - 1,
                        )
                        pixmap, effective_dpi, render_warnings = _render_pdf_page(
                            page,
                            dpi=dpi,
                            max_page_pixels=max_page_pixels,
                        )
                        render_warnings = tuple(
                            dict.fromkeys((*load_warnings, *render_warnings))
                        )
                        _, page_record = await _layout_page_async(
                            pixmap,
                            page_number,
                            target=target,
                            stage_assets=stage_assets,
                            detector=detector,
                            client=client,
                            model=selected_model,
                            prompt=selected_prompt,
                            embed_page_image=embed_page_image,
                            request_semaphore=request_semaphore,
                            layout_semaphore=layout_semaphore,
                            max_tokens=max_tokens,
                            ngram_size=ngram_size,
                            ngram_window=ngram_window,
                            max_data_url_bytes=max_data_url_bytes,
                            clean_output=clean_output,
                            requested_dpi=dpi,
                            effective_dpi=effective_dpi,
                            existing_record=(
                                existing_record
                                if isinstance(existing_record, dict)
                                else None
                            ),
                            checkpoint=checkpoint_page,
                        )
                        page_record["render_warnings"] = list(render_warnings)
                        await checkpoint_page(page_record)
                elapsed_s = perf_counter() - page_started_at
                pages_completed += 1
                if page_progress is not None:
                    page_progress(pages_completed, pages_total, elapsed_s)
                return page_number, page_record, elapsed_s

            tasks = [
                asyncio.create_task(process_page(page_number))
                for page_number in range(1, pages_total + 1)
            ]
            try:
                parsed_pages = await asyncio.gather(*tasks)
            finally:
                for task in tasks:
                    if not task.done():
                        task.cancel()
                await asyncio.gather(*tasks, return_exceptions=True)

        parsed_pages.sort(key=lambda item: item[0])
        layout["pages"] = [
            copy.deepcopy(page_record)
            for _, page_record, _ in parsed_pages
        ]
        status, regions_total, regions_completed, regions_failed = (
            await asyncio.to_thread(_publish_completed_layout, target, layout)
        )
        error = (
            f"{regions_failed} layout regions failed; see warnings and sidecar"
            if regions_failed
            else None
        )
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            cast(Literal["processed", "partial"], status),
            pages_total,
            pages_completed,
            perf_counter() - started_at,
            error,
            regions_total,
            regions_completed,
            regions_failed,
        )
    except Exception as error:
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            "failed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
            f"{type(error).__name__}: {error}",
        )
    finally:
        if owned_detector is not None:
            try:
                await asyncio.to_thread(owned_detector.stop)
            except Exception:
                pass


async def _ocr_image_layout(
    image_path: Path | str,
    markdown_path: Path | str,
    *,
    client: AsyncOpenAI,
    service_base_url: str,
    model: str | None,
    prompt: str,
    overwrite: bool,
    max_tokens: int,
    ngram_size: int,
    ngram_window: int,
    max_data_url_bytes: int,
    clean_output: bool,
    request_semaphore: asyncio.Semaphore | None,
    layout_detector: Any | None,
    layout_model: str,
    layout_threshold: float,
    layout_device: str | None,
    embed_page_image: bool,
    resume_partial: bool,
    layout_semaphore: asyncio.Semaphore | None,
) -> UnlimitedOCRResult:
    source = Path(image_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    metadata_path = target.with_suffix(".layout.json")
    selected_base_url = _resolve_base_url(service_base_url)
    selected_model = _resolve_model(model)
    selected_prompt = _resolve_prompt(prompt)
    if max_tokens <= 0:
        raise ValueError("max_tokens must be greater than zero")
    if not 0 <= layout_threshold <= 1:
        raise ValueError("layout_threshold must be between 0 and 1")
    if not layout_model.strip():
        raise ValueError("layout_model must not be empty")
    _vllm_extra_body(ngram_size, ngram_window)
    if state := _target_state(
        source,
        target,
        overwrite=overwrite,
        metadata_path=metadata_path,
    ):
        if not (
            state.status == "skipped"
            and resume_partial
            and _layout_output_is_partial(metadata_path)
        ):
            return state

    started_at = perf_counter()
    pages_total = 0
    pages_completed = 0
    owned_detector: Any | None = None
    try:
        if not source.is_file():
            raise FileNotFoundError(f"Image does not exist: {source}")
        if source.suffix.casefold() not in _IMAGE_SUFFIXES:
            raise ValueError(f"Unsupported image type: {source.suffix or '<none>'}")
        pixmap = _load_image_pixmap(source)
        pages_total = 1
        detector = layout_detector
        if detector is None:
            detector = await asyncio.to_thread(
                create_pp_doclayout_detector,
                model_name=layout_model,
                threshold=layout_threshold,
                device=layout_device,
            )
            owned_detector = detector
        target.parent.mkdir(parents=True, exist_ok=True)
        settings = _layout_settings(
            service_base_url=selected_base_url,
            model=selected_model,
            prompt=selected_prompt,
            dpi=None,
            layout_model=layout_model,
            layout_threshold=layout_threshold,
            embed_page_image=embed_page_image,
            max_tokens=max_tokens,
            ngram_size=ngram_size,
            ngram_window=ngram_window,
            max_data_url_bytes=max_data_url_bytes,
            clean_output=clean_output,
        )
        _, stage_assets, layout = _prepare_layout_workspace(
            source,
            target,
            pages_total=1,
            settings=settings,
            resume_partial=resume_partial,
            overwrite=overwrite,
        )
        checkpoint_lock = asyncio.Lock()

        async def checkpoint_page(page_record: dict[str, Any]) -> None:
            async with checkpoint_lock:
                layout["pages"][0] = copy.deepcopy(page_record)
                snapshot = copy.deepcopy(layout)
                await asyncio.to_thread(
                    _write_layout_checkpoint,
                    target,
                    snapshot,
                )

        existing_record = layout["pages"][0]
        if not (
            isinstance(existing_record, dict)
            and existing_record.get("status") == "completed"
        ):
            _, page_record = await _layout_page_async(
                pixmap,
                1,
                target=target,
                stage_assets=stage_assets,
                detector=detector,
                client=client,
                model=selected_model,
                prompt=selected_prompt,
                embed_page_image=embed_page_image,
                request_semaphore=request_semaphore,
                layout_semaphore=layout_semaphore,
                max_tokens=max_tokens,
                ngram_size=ngram_size,
                ngram_window=ngram_window,
                max_data_url_bytes=max_data_url_bytes,
                clean_output=clean_output,
                existing_record=(
                    existing_record
                    if isinstance(existing_record, dict)
                    else None
                ),
                checkpoint=checkpoint_page,
            )
            layout["pages"][0] = page_record
        pages_completed = 1
        status, regions_total, regions_completed, regions_failed = (
            await asyncio.to_thread(_publish_completed_layout, target, layout)
        )
        error = (
            f"{regions_failed} layout regions failed; see warnings and sidecar"
            if regions_failed
            else None
        )
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            cast(Literal["processed", "partial"], status),
            pages_total,
            pages_completed,
            perf_counter() - started_at,
            error,
            regions_total,
            regions_completed,
            regions_failed,
        )
    except Exception as error:
        return UnlimitedOCRResult(
            source,
            target,
            metadata_path,
            "failed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
            f"{type(error).__name__}: {error}",
        )
    finally:
        if owned_detector is not None:
            try:
                await asyncio.to_thread(owned_detector.stop)
            except Exception:
                pass


async def ocr_pdf(
    pdf_path: Path | str,
    markdown_path: Path | str,
    *,
    client: AsyncOpenAI,
    service_base_url: str = _DEFAULT_BASE_URL,
    model: str | None = None,
    prompt: str = _DEFAULT_PROMPT,
    dpi: int = 300,
    max_page_pixels: int = _DEFAULT_MAX_PAGE_PIXELS,
    overwrite: bool = False,
    page_concurrency: int = 2,
    max_tokens: int = _DEFAULT_MAX_TOKENS,
    ngram_size: int = _DEFAULT_NGRAM_SIZE,
    ngram_window: int = _DEFAULT_NGRAM_WINDOW,
    max_data_url_bytes: int = _DEFAULT_MAX_DATA_URL_BYTES,
    clean_output: bool = True,
    request_semaphore: asyncio.Semaphore | None = None,
    layout_mode: UnlimitedOCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
    resume_partial: bool = True,
    layout_semaphore: asyncio.Semaphore | None = None,
    page_started: Callable[[int], None] | None = None,
    page_progress: Callable[[int, int, float], None] | None = None,
) -> UnlimitedOCRResult:
    """Parse one PDF, reducing DPI only when a page exceeds the pixel cap."""
    source = Path(pdf_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    output_lock = _exclusive_output_lock(
        target.parent / f".{target.stem}.unlimited-ocr.lock",
        str(target),
    )
    output_lock.__enter__()
    try:
        common = {
            "client": client,
            "service_base_url": service_base_url,
            "model": model,
            "prompt": prompt,
            "overwrite": overwrite,
            "max_tokens": max_tokens,
            "ngram_size": ngram_size,
            "ngram_window": ngram_window,
            "max_data_url_bytes": max_data_url_bytes,
            "clean_output": clean_output,
            "request_semaphore": request_semaphore,
        }
        if selected_layout_mode == "plain":
            return await _ocr_pdf_plain(
                source,
                target,
                dpi=dpi,
                max_page_pixels=max_page_pixels,
                page_concurrency=page_concurrency,
                page_started=page_started,
                page_progress=page_progress,
                **common,
            )
        return await _ocr_pdf_layout(
            source,
            target,
            dpi=dpi,
            max_page_pixels=max_page_pixels,
            page_concurrency=page_concurrency,
            layout_detector=layout_detector,
            layout_model=layout_model,
            layout_threshold=layout_threshold,
            layout_device=layout_device,
            embed_page_image=embed_page_image,
            resume_partial=resume_partial,
            layout_semaphore=layout_semaphore,
            page_started=page_started,
            page_progress=page_progress,
            **common,
        )
    finally:
        output_lock.__exit__(None, None, None)


async def ocr_image(
    image_path: Path | str,
    markdown_path: Path | str,
    *,
    client: AsyncOpenAI,
    service_base_url: str = _DEFAULT_BASE_URL,
    model: str | None = None,
    prompt: str = _DEFAULT_PROMPT,
    overwrite: bool = False,
    max_tokens: int = _DEFAULT_MAX_TOKENS,
    ngram_size: int = _DEFAULT_NGRAM_SIZE,
    ngram_window: int = _DEFAULT_NGRAM_WINDOW,
    max_data_url_bytes: int = _DEFAULT_MAX_DATA_URL_BYTES,
    clean_output: bool = True,
    request_semaphore: asyncio.Semaphore | None = None,
    layout_mode: UnlimitedOCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
    resume_partial: bool = True,
    layout_semaphore: asyncio.Semaphore | None = None,
) -> UnlimitedOCRResult:
    """Parse one image with whole-page or durable PP-DocLayout reconstruction."""
    source = Path(image_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    output_lock = _exclusive_output_lock(
        target.parent / f".{target.stem}.unlimited-ocr.lock",
        str(target),
    )
    output_lock.__enter__()
    try:
        common = {
            "client": client,
            "service_base_url": service_base_url,
            "model": model,
            "prompt": prompt,
            "overwrite": overwrite,
            "max_tokens": max_tokens,
            "ngram_size": ngram_size,
            "ngram_window": ngram_window,
            "max_data_url_bytes": max_data_url_bytes,
            "clean_output": clean_output,
            "request_semaphore": request_semaphore,
        }
        if selected_layout_mode == "plain":
            return await _ocr_image_plain(source, target, **common)
        return await _ocr_image_layout(
            source,
            target,
            layout_detector=layout_detector,
            layout_model=layout_model,
            layout_threshold=layout_threshold,
            layout_device=layout_device,
            embed_page_image=embed_page_image,
            resume_partial=resume_partial,
            layout_semaphore=layout_semaphore,
            **common,
        )
    finally:
        output_lock.__exit__(None, None, None)

In [ ]:
# | export
class _UnlimitedOCRFolderProgress:
    """Render page/file progress and completed-file history."""

    def __init__(
        self,
        total_files: int,
        description: str,
        enabled: bool,
        initial_results: Sequence[tuple[int, UnlimitedOCRResult]] = (),
    ) -> None:
        self._total_files = total_files
        self._files_completed = 0
        self._description = description
        self._enabled = enabled
        self._closed = False
        self._active_order: list[int] = []
        self._active: dict[int, _ActivePageProgress] = {}
        self._history: dict[
            int, tuple[UnlimitedOCRResult, float | None]
        ] = {}
        self._displayed_index: int | None = None
        self._notebook_mode = enabled and _running_in_notebook()
        self._html_factory: Callable[[str], Any] | None = None
        self._display: Callable[..., Any] | None = None
        self._display_handle: Any = None
        self._page_bar: Any = None
        self._file_bar: Any = None
        for index, result in initial_results:
            self._store_result(index, result, None)

        if self._notebook_mode:
            from IPython.display import HTML, display

            self._html_factory = HTML
            self._display = display
            self._refresh_notebook()
        elif enabled:
            self._page_bar = tqdm(
                total=None,
                desc="Unlimited-OCR pages: waiting",
                unit="page",
                position=0,
                leave=False,
                dynamic_ncols=True,
            )
            self._file_bar = tqdm(
                total=total_files,
                desc=description,
                unit="file",
                position=1,
                leave=True,
                dynamic_ncols=True,
            )

    @property
    def notebook_mode(self) -> bool:
        """Whether this renderer owns an updatable notebook display."""
        return self._notebook_mode

    @staticmethod
    def _source_modified_ns(source_path: Path) -> int:
        try:
            return source_path.stat().st_mtime_ns
        except OSError:
            return -1

    def _store_result(
        self,
        index: int,
        result: UnlimitedOCRResult,
        elapsed_s: float | None,
    ) -> None:
        self._history[index] = (result, elapsed_s)

    def record_result(
        self,
        index: int,
        result: UnlimitedOCRResult,
        elapsed_s: float | None = None,
    ) -> None:
        """Add a skipped or completed result below the progress bars."""
        self._store_result(index, result, elapsed_s)
        if self._notebook_mode:
            self._refresh_notebook()

    def start_file(
        self, index: int, source_path: Path, pages_total: int | None = None
    ) -> None:
        self._active[index] = _ActivePageProgress(source_path, pages_total)
        self._active_order.append(index)
        self._refresh_page_bar()

    def set_pages_total(self, index: int, pages_total: int) -> None:
        state = self._active.get(index)
        if state is not None:
            state.pages_total = pages_total
            self._refresh_page_bar()

    def complete_page(
        self,
        index: int,
        pages_completed: int,
        pages_total: int,
        elapsed_s: float,
    ) -> None:
        state = self._active.get(index)
        if state is not None:
            state.pages_total = pages_total
            state.pages_completed = pages_completed
            state.last_page_elapsed_s = elapsed_s
            self._refresh_page_bar()

    def finish_file(self, index: int) -> None:
        self._active.pop(index, None)
        if index in self._active_order:
            self._active_order.remove(index)
        self._files_completed += 1
        if self._file_bar is not None:
            self._file_bar.update(1)
        self._refresh_page_bar()

    def _refresh_page_bar(self) -> None:
        if not self._enabled:
            return
        if self._notebook_mode:
            self._refresh_notebook()
            return
        if self._page_bar is None:
            return
        if not self._active_order:
            self._displayed_index = None
            self._page_bar.clear()
            self._page_bar.set_description_str(
                "Unlimited-OCR pages: waiting", refresh=False
            )
            self._page_bar.total = None
            self._page_bar.n = 0
            self._page_bar.set_postfix_str("", refresh=False)
            self._page_bar.refresh()
            return

        newest_index = self._active_order[-1]
        state = self._active[newest_index]
        if newest_index != self._displayed_index:
            self._page_bar.clear()
            self._displayed_index = newest_index
            self._page_bar.n = 0
            self._page_bar.last_print_n = 0
            now = self._page_bar._time()
            self._page_bar.start_t = now
            self._page_bar.last_print_t = now
        self._page_bar.set_description_str(
            f"Unlimited-OCR pages: {state.source_path}", refresh=False
        )
        self._page_bar.total = state.pages_total
        self._page_bar.n = state.pages_completed
        elapsed = state.last_page_elapsed_s
        self._page_bar.set_postfix_str(
            f"last={elapsed:.2f}s" if elapsed is not None else "",
            refresh=False,
        )
        self._page_bar.refresh()

    @staticmethod
    def _html_progress_row(
        label: str,
        completed: int,
        total: int | None,
        detail: str = "",
    ) -> str:
        count = f"{completed}/{total}" if total is not None else f"{completed}/?"
        percentage = 100 * completed / total if total else 0
        progress = (
            f'<progress value="{min(completed, total)}" max="{total}" '
            'style="width:100%;height:0.8rem"></progress>'
            if total is not None
            else '<progress style="width:100%;height:0.8rem"></progress>'
        )
        suffix = f" &nbsp; {escape(detail)}" if detail else ""
        return (
            '<div style="margin:0 0 0.45rem 0">'
            '<div style="display:flex;gap:1rem;justify-content:space-between;">'
            f'<span style="overflow-wrap:anywhere">{escape(label)}</span>'
            f'<span style="white-space:nowrap">{percentage:.0f}% &nbsp; '
            f"{count}{suffix}</span></div>{progress}</div>"
        )

    def _notebook_history_html(self) -> str:
        if not self._history:
            return ""
        dated_results = [
            (self._source_modified_ns(result.source_path), result, elapsed_s)
            for result, elapsed_s in self._history.values()
        ]
        ordered = sorted(
            dated_results,
            key=lambda item: (
                -item[0],
                item[1].source_path.as_posix().casefold(),
            ),
        )
        entries: list[str] = []
        for modified_ns, result, elapsed_s in ordered:
            modified = (
                datetime.fromtimestamp(modified_ns / 1_000_000_000)
                .astimezone()
                .isoformat(timespec="seconds")
                if modified_ns >= 0
                else "unknown"
            )
            elapsed = (
                f" &nbsp; elapsed={elapsed_s:.2f}s"
                if elapsed_s is not None
                else ""
            )
            entries.append(
                '<div style="display:flex;gap:1rem;justify-content:space-between;'
                'padding:0.12rem 0;border-top:1px solid rgba(127,127,127,.18)">'
                f'<span style="overflow-wrap:anywhere"><strong>{escape(result.status)}</strong> '
                f"{escape(str(result.source_path))}</span>"
                f'<span style="white-space:nowrap">modified={escape(modified)}'
                f"{elapsed}</span></div>"
            )
        return (
            '<div style="margin-top:0.65rem">'
            f'<div><strong>Unlimited-OCR files (newest to oldest, '
            f'{len(ordered)} shown)</strong></div>'
            + "".join(entries)
            + "</div>"
        )

    def _notebook_html(self) -> str:
        if self._closed:
            page_row = self._html_progress_row(
                "Unlimited-OCR pages: finished", 1, 1
            )
        elif self._active_order:
            state = self._active[self._active_order[-1]]
            elapsed = state.last_page_elapsed_s
            detail = f"last={elapsed:.2f}s" if elapsed is not None else ""
            page_row = self._html_progress_row(
                f"Unlimited-OCR pages: {state.source_path}",
                state.pages_completed,
                state.pages_total,
                detail,
            )
        else:
            page_row = self._html_progress_row(
                "Unlimited-OCR pages: waiting", 0, None
            )
        file_row = self._html_progress_row(
            self._description, self._files_completed, self._total_files
        )
        return (
            '<div style="font-family:var(--vscode-editor-font-family,monospace);'
            'font-size:var(--vscode-editor-font-size,13px);padding:0.25rem 0">'
            + page_row
            + file_row
            + self._notebook_history_html()
            + "</div>"
        )

    def _refresh_notebook(self) -> None:
        if self._html_factory is None or self._display is None:
            return
        content = self._html_factory(self._notebook_html())
        if self._display_handle is None:
            self._display_handle = self._display(content, display_id=True)
        else:
            self._display_handle.update(content)

    def close(self) -> None:
        if self._notebook_mode:
            self._active.clear()
            self._active_order.clear()
            self._closed = True
            self._refresh_notebook()
            return
        if self._page_bar is not None:
            self._page_bar.close()
        if self._file_bar is not None:
            self._file_bar.close()


def _print_unlimited_ocr_file_report(
    results: Sequence[UnlimitedOCRResult],
) -> None:
    """Print discovered sources by modification time, newest first."""
    dated_results = [
        (_UnlimitedOCRFolderProgress._source_modified_ns(result.source_path), result)
        for result in results
    ]
    dated_results.sort(
        key=lambda item: (
            -item[0],
            item[1].source_path.as_posix().casefold(),
        )
    )
    print(
        f"Unlimited-OCR files (newest to oldest, {len(dated_results)} total):"
    )
    for modified_ns, result in dated_results:
        modified = (
            datetime.fromtimestamp(modified_ns / 1_000_000_000)
            .astimezone()
            .isoformat(timespec="seconds")
            if modified_ns >= 0
            else "unknown"
        )
        elapsed = (
            f" | elapsed={result.elapsed_s:.2f}s" if result.elapsed_s > 0 else ""
        )
        print(
            f"Unlimited-OCR file: {result.source_path} | "
            f"status={result.status} | modified={modified}{elapsed}"
        )


@_ocr_folder_workflow
async def ocr_folder(
    root_folder: Path | str,
    *,
    base_url: str | None = None,
    api_key: str | None = None,
    model: str | None = None,
    output_dir_name: str = ".md_unlimited",
    prompt: str = _DEFAULT_PROMPT,
    dpi: int = 300,
    max_page_pixels: int = _DEFAULT_MAX_PAGE_PIXELS,
    office_converter: Path | str | None = None,
    office_conversion_timeout_s: float = _DEFAULT_OFFICE_CONVERSION_TIMEOUT_S,
    overwrite: bool = False,
    request_timeout_s: float = 1200,
    max_concurrency: int | None = None,
    page_concurrency: int | None = None,
    max_tokens: int = _DEFAULT_MAX_TOKENS,
    ngram_size: int = _DEFAULT_NGRAM_SIZE,
    ngram_window: int = _DEFAULT_NGRAM_WINDOW,
    max_data_url_bytes: int = _DEFAULT_MAX_DATA_URL_BYTES,
    clean_output: bool = True,
    show_progress: bool = True,
    client: AsyncOpenAI | None = None,
    layout_mode: UnlimitedOCRLayoutMode = "plain",
    layout_detector: Any | None = None,
    layout_model: str = _DEFAULT_LAYOUT_MODEL,
    layout_threshold: float = 0.3,
    layout_device: str | None = None,
    embed_page_image: bool = False,
    resume_partial: bool = True,
) -> list[UnlimitedOCRResult]:
    """Parse a folder with whole-page or durable PP-DocLayout reconstruction.

    When progress is enabled, the active page appears above aggregate file
    progress. Notebook displays also keep a live, newest-first history of
    skipped and completed files; terminals print the equivalent final report.
    Visible PPT/PPTX/DOC/DOCX sources are first converted to current sibling
    PDFs with LibreOffice, then processed through the normal PDF OCR path.
    """
    root = _resolve_root(root_folder)
    selected_output_dir = _resolve_output_dir_name(output_dir_name)
    selected_base_url = _resolve_base_url(base_url)
    selected_model = _resolve_model(model)
    selected_prompt = _resolve_prompt(prompt)
    selected_layout_mode = _resolve_layout_mode(layout_mode)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if max_page_pixels <= 0:
        raise ValueError("max_page_pixels must be greater than zero")
    if office_conversion_timeout_s <= 0:
        raise ValueError("office_conversion_timeout_s must be greater than zero")
    if request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be greater than zero")
    if max_tokens <= 0:
        raise ValueError("max_tokens must be greater than zero")
    if not 0 <= layout_threshold <= 1:
        raise ValueError("layout_threshold must be between 0 and 1")
    if selected_layout_mode == "pp-doclayout" and not layout_model.strip():
        raise ValueError("layout_model must not be empty")
    _vllm_extra_body(ngram_size, ngram_window)
    selected_max_concurrency = _positive_int(
        max_concurrency,
        "UNLIMITED_OCR_MAX_CONCURRENCY",
        4,
    )
    selected_page_concurrency = _positive_int(
        page_concurrency,
        "UNLIMITED_OCR_PAGE_CONCURRENCY",
        2,
    )

    await _convert_office_sources_to_pdfs(
        root,
        selected_output_dir,
        converter=office_converter,
        timeout_s=office_conversion_timeout_s,
        show_progress=show_progress,
    )
    jobs = _ocr_jobs(root, selected_output_dir)
    if not jobs:
        print(f"No supported PDF, Office, or image files found under {root}")
        return []

    results_by_index: dict[int, UnlimitedOCRResult] = {}
    pending_jobs: list[tuple[int, tuple[Path, Path, Path]]] = []
    for index, (source_path, markdown_path, metadata_path) in enumerate(jobs):
        selected_metadata_path = (
            markdown_path.with_suffix(".layout.json")
            if selected_layout_mode == "pp-doclayout"
            else metadata_path
        )
        if not overwrite and markdown_path.is_file():
            results_by_index[index] = UnlimitedOCRResult(
                source_path,
                markdown_path,
                selected_metadata_path,
                "skipped",
            )
        else:
            pending_jobs.append(
                (index, (source_path, markdown_path, selected_metadata_path))
            )

    if not pending_jobs:
        results = [results_by_index[index] for index in range(len(jobs))]
        print(
            "Unlimited-OCR complete: "
            f"0 processed, 0 partial, {len(results)} skipped, 0 failed"
        )
        _print_unlimited_ocr_file_report(results)
        return results

    output_lock = _exclusive_output_lock(
        root / ".unlimited-ocr-folder.lock",
        f"Unlimited-OCR output tree {root / selected_output_dir}",
    )
    output_lock.__enter__()
    try:
        parser_client = (
            client
            if client is not None
            else _create_client(selected_base_url, api_key, request_timeout_s)
        )
    except BaseException:
        output_lock.__exit__(None, None, None)
        raise

    active_layout_detector = layout_detector
    owned_layout_detector = False
    if selected_layout_mode == "pp-doclayout" and active_layout_detector is None:
        try:
            active_layout_detector = await asyncio.to_thread(
                create_pp_doclayout_detector,
                model_name=layout_model,
                threshold=layout_threshold,
                device=layout_device,
            )
            owned_layout_detector = True
        except BaseException:
            if client is None:
                await parser_client.close()
            output_lock.__exit__(None, None, None)
            raise

    file_semaphore = asyncio.Semaphore(selected_max_concurrency)
    request_semaphore = asyncio.Semaphore(selected_max_concurrency)
    layout_semaphore = asyncio.Semaphore(1)
    description = (
        f"Unlimited-OCR files ({selected_base_url}, "
        f"layout={selected_layout_mode}, "
        f"files/requests={selected_max_concurrency}, "
        f"pages/PDF={selected_page_concurrency})"
    )
    progress = _UnlimitedOCRFolderProgress(
        len(pending_jobs),
        description,
        show_progress,
        initial_results=tuple(results_by_index.items()),
    )

    async def run_job(
        index: int, job: tuple[Path, Path, Path]
    ) -> tuple[int, UnlimitedOCRResult, float]:
        source_path, markdown_path, metadata_path = job
        async with file_semaphore:
            started_at = perf_counter()
            is_pdf = source_path.suffix.casefold() == ".pdf"
            progress.start_file(index, source_path, None if is_pdf else 1)

            def report_pages_total(pages_total: int) -> None:
                progress.set_pages_total(index, pages_total)

            def report_page(
                pages_completed: int,
                pages_total: int,
                elapsed_s: float,
            ) -> None:
                progress.complete_page(
                    index, pages_completed, pages_total, elapsed_s
                )

            try:
                common = {
                    "client": parser_client,
                    "service_base_url": selected_base_url,
                    "model": selected_model,
                    "prompt": selected_prompt,
                    "overwrite": overwrite,
                    "max_tokens": max_tokens,
                    "ngram_size": ngram_size,
                    "ngram_window": ngram_window,
                    "max_data_url_bytes": max_data_url_bytes,
                    "clean_output": clean_output,
                    "request_semaphore": request_semaphore,
                    "layout_mode": selected_layout_mode,
                    "layout_detector": active_layout_detector,
                    "layout_model": layout_model,
                    "layout_threshold": layout_threshold,
                    "layout_device": layout_device,
                    "embed_page_image": embed_page_image,
                    "resume_partial": resume_partial,
                    "layout_semaphore": layout_semaphore,
                }
                if is_pdf:
                    result = await ocr_pdf(
                        source_path,
                        markdown_path,
                        dpi=dpi,
                        max_page_pixels=max_page_pixels,
                        page_concurrency=selected_page_concurrency,
                        page_started=(
                            report_pages_total if show_progress else None
                        ),
                        page_progress=report_page if show_progress else None,
                        **common,
                    )
                else:
                    result = await ocr_image(
                        source_path,
                        markdown_path,
                        **common,
                    )
                    if (
                        result.status in {"processed", "partial"}
                        and show_progress
                    ):
                        progress.complete_page(
                            index,
                            result.pages_completed,
                            result.pages_total,
                            result.elapsed_s,
                        )
            except Exception as error:
                result = UnlimitedOCRResult(
                    source_path,
                    markdown_path,
                    metadata_path,
                    "failed",
                    error=f"{type(error).__name__}: {error}",
                )
            finally:
                progress.finish_file(index)
            return index, result, perf_counter() - started_at

    @asynccontextmanager
    async def execution() -> AsyncIterator[
        _OCRFolderExecution[tuple[Path, Path, Path], UnlimitedOCRResult]
    ]:
        try:
            yield _OCRFolderExecution(progress=progress, run_job=run_job)
        finally:
            try:
                if owned_layout_detector and active_layout_detector is not None:
                    try:
                        await asyncio.to_thread(active_layout_detector.stop)
                    except Exception:
                        pass
            finally:
                try:
                    if client is None:
                        await parser_client.close()
                finally:
                    output_lock.__exit__(None, None, None)

    return cast(
        Any,
        _OCRFolderPlan(
            operation_name="Unlimited-OCR",
            job_count=len(jobs),
            pending_jobs=pending_jobs,
            initial_results=results_by_index,
            execution=execution,
            report_results=_print_unlimited_ocr_file_report,
        ),
    )

## Configuration and batch execution

The health endpoint at the default host does not require authentication, but
its `/v1` API does. Add the bearer token to the project `.env`:

```dotenv
UNLIMITED_OCR_API_KEY=replace-with-the-service-token
# Optional when the deployment uses a served-model alias:
# UNLIMITED_OCR_MODEL=Unlimited-OCR
```

Run `list_served_models()` first to confirm the model ID. Structured mode
requires `uv sync --extra ocr-layout`. It writes durable checkpoints after
layout detection and every completed region, resumes compatible interrupted
work when `RESUME_PARTIAL=True`, and atomically publishes Markdown,
`.layout.json`, and `.assets/`. The real OCR calls below are opt-in because
they can be long-running and write generated files.

In [ ]:
from pathlib import Path

PDF_ROOT = PROJ_ROOT / "res" / "PDF-20260721"
SERVICE_BASE_URL = os.getenv(
    "UNLIMITED_OCR_BASE_URL",
    _DEFAULT_BASE_URL,
)
OCR_API_KEY = os.getenv("UNLIMITED_OCR_API_KEY") or None
OCR_MODEL = os.getenv("UNLIMITED_OCR_MODEL") or None
OUTPUT_DIR_NAME = ".md_unlimited"
OCR_DPI = 300
# Preserve OCR_DPI for normal pages and downscale only oversized pages.
MAX_PAGE_PIXELS = int(
    os.getenv("UNLIMITED_OCR_MAX_PAGE_PIXELS", str(_DEFAULT_MAX_PAGE_PIXELS))
)
REQUEST_TIMEOUT_S = 1200
MAX_CONCURRENCY = int(os.getenv("UNLIMITED_OCR_MAX_CONCURRENCY", "4"))
PAGE_CONCURRENCY = int(os.getenv("UNLIMITED_OCR_PAGE_CONCURRENCY", "2"))
MAX_TOKENS = 8192
OCR_LAYOUT_MODE: UnlimitedOCRLayoutMode = "pp-doclayout"
LAYOUT_MODEL = _DEFAULT_LAYOUT_MODEL
LAYOUT_THRESHOLD = 0.3
LAYOUT_DEVICE: str | None = None
EMBED_PAGE_IMAGE = False
RESUME_PARTIAL = True
OVERWRITE = False

In [ ]:
# | notest
# Confirm credentials and the served model name:
await list_served_models(
    base_url=SERVICE_BASE_URL,
    api_key=OCR_API_KEY,
)

In [ ]:
# | notest
# Run the recursive batch:
results = await ocr_folder(
    PDF_ROOT,
    base_url=SERVICE_BASE_URL,
    api_key=OCR_API_KEY,
    model=OCR_MODEL,
    output_dir_name=OUTPUT_DIR_NAME,
    dpi=OCR_DPI,
    max_page_pixels=MAX_PAGE_PIXELS,
    overwrite=OVERWRITE,
    request_timeout_s=REQUEST_TIMEOUT_S,
    max_concurrency=MAX_CONCURRENCY,
    page_concurrency=PAGE_CONCURRENCY,
    max_tokens=MAX_TOKENS,
    layout_mode=OCR_LAYOUT_MODE,
    layout_model=LAYOUT_MODEL,
    layout_threshold=LAYOUT_THRESHOLD,
    layout_device=LAYOUT_DEVICE,
    embed_page_image=EMBED_PAGE_IMAGE,
    resume_partial=RESUME_PARTIAL,
)

## Tests

Tests use small generated documents and a fake OpenAI-compatible client. They
make no network calls and do not write to the repository.

In [ ]:
# | hide
from contextlib import redirect_stderr, redirect_stdout
from io import StringIO
from types import SimpleNamespace
from unittest.mock import patch

from fastcore.test import test_eq


class FakeAsyncOpenAIClient:
    def __init__(self, responses=(), delay_s=0.001):
        self.responses = list(responses)
        self.delay_s = delay_s
        self.calls = []
        self.active_calls = 0
        self.max_active_calls = 0
        self.closed = False
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create)
        )

    async def _create(self, **kwargs):
        self.calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat completion call")
        response = self.responses.pop(0)
        delay_s = self.delay_s
        if isinstance(response, tuple):
            delay_s, response = response
        self.active_calls += 1
        self.max_active_calls = max(self.max_active_calls, self.active_calls)
        try:
            await asyncio.sleep(delay_s)
        finally:
            self.active_calls -= 1
        if isinstance(response, Exception):
            raise response
        if not isinstance(response, str):
            return response
        call_number = len(self.calls)
        return SimpleNamespace(
            id=f"response-{call_number}",
            choices=[
                SimpleNamespace(
                    message=SimpleNamespace(content=response),
                    finish_reason="stop",
                )
            ],
            usage=SimpleNamespace(
                prompt_tokens=100 + call_number,
                completion_tokens=10 + call_number,
                total_tokens=110 + 2 * call_number,
            ),
        )

    async def close(self):
        self.closed = True


class FakeLayoutDetector:
    def __init__(self, regions):
        self.regions = regions
        self.calls = []
        self.stopped = False

    def process(self, images, save_visualization=False):
        self.calls.append((images, save_visualization))
        return [self.regions], None

    def stop(self):
        self.stopped = True


def make_pdf(path: Path, labels=("page",), password: str | None = None):
    document = pymupdf.open()
    for label in labels:
        page = document.new_page(width=240, height=120)
        page.insert_text((24, 60), label)
    if password is None:
        document.save(path)
    else:
        document.save(
            path,
            encryption=pymupdf.PDF_ENCRYPT_AES_256,
            owner_pw="owner-password",
            user_pw=password,
        )
    document.close()


def make_image(path: Path, label: str = "image"):
    document = pymupdf.open()
    page = document.new_page(width=240, height=120)
    page.insert_text((24, 60), label)
    page.get_pixmap(alpha=False).save(path)
    document.close()

In [ ]:
# | hide
def test_configuration_and_grounding():
    test_eq(_resolve_base_url("172.27.74.16:7870"), "http://172.27.74.16:7870/v1")
    test_eq(
        _resolve_base_url("http://172.27.74.16:7870/v1/"),
        "http://172.27.74.16:7870/v1",
    )
    test_eq(_resolve_model(None), "baidu/Unlimited-OCR")
    test_eq(
        clean_grounding(
            "<|ref|># Heading<|/ref|><|det|>[[1,2,3,4]]<|/det|>\n\nBody"
        ),
        "# Heading\n\nBody",
    )
    test_eq(
        _vllm_extra_body(35, 128),
        {
            "skip_special_tokens": False,
            "vllm_xargs": {"ngram_size": 35, "window_size": 128},
        },
    )
    recovered, recovery = _validated_layout_content(
        "Stable heading\n" + "```\n" * 20,
        finish_reason="length",
        task_type="text",
        label="paragraph_title",
    )
    test_eq(recovered, "Stable heading")
    assert recovery
    try:
        _validated_layout_content(
            "Incomplete table",
            finish_reason="length",
            task_type="table",
            label="table",
        )
    except ValueError as error:
        assert "output-token limit" in str(error)
    else:
        raise AssertionError("Expected incomplete length-limited table failure")
    for invalid in ("", "document parsing.", " <imagex>bad"):
        try:
            _resolve_prompt(invalid)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Expected invalid prompt: {invalid!r}")


test_configuration_and_grounding()

In [ ]:
# | hide
async def test_pdf_payload_order_and_sidecar():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "two-pages.pdf"
        target = root / "two-pages.md"
        make_pdf(source, ("one", "two"))
        first = (
            "<|ref|># One<|/ref|>"
            "<|det|>[[0,0,999,100]]<|/det|>\n\nFirst body"
        )
        second = "<|ref|>## Two<|/ref|><|det|>[[0,0,999,100]]<|/det|>"
        client = FakeAsyncOpenAIClient(((0.03, first), (0.005, second)))
        result = await ocr_pdf(
            source,
            target,
            client=client,
            dpi=72,
            page_concurrency=2,
        )
        assert result.status == "processed", result.error
        test_eq(result.pages_total, 2)
        test_eq(result.pages_completed, 2)
        test_eq(
            target.read_text(encoding="utf-8"),
            "<!-- Page 1 -->\n\n# One\n\nFirst body\n\n"
            "<!-- Page 2 -->\n\n## Two\n",
        )
        assert client.max_active_calls == 2

        for call in client.calls:
            test_eq(call["model"], "baidu/Unlimited-OCR")
            test_eq(call["max_tokens"], 8192)
            test_eq(call["temperature"], 0.0)
            test_eq(
                call["extra_body"],
                {
                    "skip_special_tokens": False,
                    "vllm_xargs": {"ngram_size": 35, "window_size": 128},
                },
            )
            content = call["messages"][0]["content"]
            test_eq(content[0], {"type": "text", "text": "<image>document parsing."})
            image_url = content[1]["image_url"]["url"]
            assert image_url.startswith("data:image/png;base64,")
            assert base64.b64decode(image_url.split(",", 1)[1]).startswith(b"\x89PNG")

        metadata = json.loads(
            target.with_suffix(".unlimited.json").read_text(encoding="utf-8")
        )
        test_eq(metadata["provider"], "baidu/Unlimited-OCR")
        test_eq(metadata["service_base_url"], "http://172.27.74.16:7870/v1")
        test_eq(metadata["pages"][0]["raw_content"], first)
        test_eq(metadata["pages"][1]["content"], "## Two")
        test_eq(metadata["pages"][0]["effective_dpi"], 72)
        assert "api_key" not in metadata

    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "adaptive.pdf"
        target = root / "adaptive.md"
        document = pymupdf.open()
        document.new_page(width=100, height=100).insert_text((10, 50), "small")
        document.new_page(width=1000, height=1000).insert_text((10, 50), "large")
        document.save(source)
        document.close()

        result = await ocr_pdf(
            source,
            target,
            client=FakeAsyncOpenAIClient(("small", "large")),
            dpi=72,
            max_page_pixels=100_000,
            page_concurrency=1,
        )
        assert result.status == "processed", result.error
        metadata = json.loads(
            target.with_suffix(".unlimited.json").read_text(encoding="utf-8")
        )
        test_eq(metadata["max_page_pixels"], 100_000)
        test_eq(metadata["pages"][0]["effective_dpi"], 72)
        assert metadata["pages"][1]["effective_dpi"] < 72
        assert all(
            page["render_pixels"] <= 100_000 for page in metadata["pages"]
        )


await test_pdf_payload_order_and_sidecar()

In [ ]:
# | hide
def test_ocr_jobs_ignore_hidden_folders():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        visible = root / "visible"
        visible.mkdir()
        hidden_directories = [root / ".hidden", visible / ".cache"]
        for directory in hidden_directories:
            directory.mkdir()
        make_image(root / "root.png")
        make_image(visible / "nested.png")
        (visible / "slides.PPTX").write_bytes(b"visible office source")
        (root / "~$draft.docx").write_bytes(b"temporary office source")
        for directory in hidden_directories:
            make_image(directory / "ignored.png")
            (directory / "ignored.docx").write_bytes(b"hidden office source")

        jobs = _ocr_jobs(root, ".md_unlimited")
        office_sources = _office_sources(root, ".md_unlimited")

    test_eq(
        [source.relative_to(root).as_posix() for source, _, _ in jobs],
        ["root.png", "visible/nested.png"],
    )
    test_eq(
        [source.relative_to(root).as_posix() for source in office_sources],
        ["visible/slides.PPTX"],
    )


test_ocr_jobs_ignore_hidden_folders()


async def test_office_sources_convert_and_follow_pdf_ocr_path():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        office_source = root / "proposal.docx"
        office_source.write_bytes(b"synthetic Office source")
        conversion_calls = []

        def fake_convert(source, *, converter, timeout_s):
            conversion_calls.append((source, converter, timeout_s))
            target = source.with_suffix(".pdf")
            make_pdf(target, labels=("converted Office page",))
            return target

        client = FakeAsyncOpenAIClient(("# Converted Office document",))
        with (
            patch(
                __name__ + "._resolve_office_converter",
                return_value="/fake/libreoffice",
            ),
            patch(__name__ + "._convert_office_to_pdf", side_effect=fake_convert),
        ):
            results = await ocr_folder(
                root,
                client=client,
                max_concurrency=1,
                page_concurrency=1,
                show_progress=False,
            )

        sibling_pdf = root / "proposal.pdf"
        assert sibling_pdf.is_file()
        test_eq(len(conversion_calls), 1)
        test_eq(conversion_calls[0][0], office_source)
        test_eq(conversion_calls[0][1], "/fake/libreoffice")
        test_eq([result.source_path for result in results], [sibling_pdf])
        test_eq([result.status for result in results], ["processed"])
        assert (root / ".md_unlimited" / "proposal.md").is_file()
        test_eq(len(client.calls), 1)

        with (
            patch(
                __name__ + "._resolve_office_converter",
                side_effect=AssertionError("current PDF should be reused"),
            ),
            patch(
                __name__ + "._convert_office_to_pdf",
                side_effect=AssertionError("current PDF should not be converted"),
            ),
        ):
            reused = await _convert_office_sources_to_pdfs(
                root,
                ".md_unlimited",
                converter=None,
                timeout_s=30,
                show_progress=False,
            )
        test_eq(reused, [sibling_pdf])


async def test_image_skip_failure_and_folder_concurrency():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "图像.png"
        target = root / "结果.md"
        make_image(source)
        first = await ocr_image(
            source,
            target,
            client=FakeAsyncOpenAIClient(("recognized",)),
        )
        test_eq(first.status, "processed")
        original_markdown = target.read_bytes()
        original_metadata = target.with_suffix(".unlimited.json").read_bytes()

        skipped = await ocr_image(
            source,
            target,
            client=FakeAsyncOpenAIClient(),
        )
        test_eq(skipped.status, "skipped")

        failed = await ocr_image(
            source,
            target,
            client=FakeAsyncOpenAIClient((RuntimeError("service failure"),)),
            overwrite=True,
        )
        test_eq(failed.status, "failed")
        test_eq(target.read_bytes(), original_markdown)
        test_eq(target.with_suffix(".unlimited.json").read_bytes(), original_metadata)

    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        for name in ("a.png", "b.png", "c.png", "skipped.png"):
            make_image(root / name, name)
        output_root = root / ".md_unlimited"
        output_root.mkdir()
        (output_root / "skipped.md").write_text("existing", encoding="utf-8")
        client = FakeAsyncOpenAIClient(
            ((0.02, "A"), RuntimeError("B failed"), (0.02, "C"))
        )
        output = StringIO()
        with redirect_stdout(output):
            results = await ocr_folder(
                root,
                client=client,
                max_concurrency=2,
                page_concurrency=2,
                show_progress=False,
            )
        test_eq(
            [result.status for result in results],
            ["processed", "failed", "processed", "skipped"],
        )
        assert 1 < client.max_active_calls <= 2
        test_eq(len(client.calls), 3)
        assert (output_root / "a.md").is_file()
        assert not (output_root / "b.md").exists()
        assert (output_root / "c.unlimited.json").is_file()
        assert "1 skipped" in output.getvalue()
        assert "status=processed" in output.getvalue()
        assert "status=skipped" in output.getvalue()
        assert not client.closed


def test_notebook_progress_renders_page_file_and_history_rows():
    updates: list[str] = []

    class FakeDisplayHandle:
        def update(self, content):
            updates.append(content.data)

    def fake_display(content, *, display_id):
        test_eq(display_id, True)
        updates.append(content.data)
        return FakeDisplayHandle()

    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        older_source = root / "older.pdf"
        newest_source = root / "newest.pdf"
        older_source.touch()
        newest_source.touch()
        old_mtime_ns = 1_700_000_000_000_000_000
        new_mtime_ns = old_mtime_ns + 1_000_000_000
        os.utime(older_source, ns=(old_mtime_ns, old_mtime_ns))
        os.utime(newest_source, ns=(new_mtime_ns, new_mtime_ns))
        skipped = UnlimitedOCRResult(
            older_source,
            root / "older.md",
            root / "older.unlimited.json",
            "skipped",
        )
        processed = UnlimitedOCRResult(
            newest_source,
            root / "newest.md",
            root / "newest.unlimited.json",
            "processed",
        )

        with (
            patch(__name__ + "._running_in_notebook", return_value=True),
            patch("IPython.display.display", side_effect=fake_display),
        ):
            progress = _UnlimitedOCRFolderProgress(
                1,
                "Unlimited-OCR files (test)",
                True,
                initial_results=((0, skipped),),
            )
            progress.start_file(1, newest_source, 2)
            progress.complete_page(1, 1, 2, 0.25)
            live_html = updates[-1]
            progress.finish_file(1)
            progress.record_result(1, processed, 0.25)
            progress.close()
            final_html = updates[-1]

    assert f"Unlimited-OCR pages: {newest_source}" in live_html
    assert "Unlimited-OCR files (test)" in live_html
    assert live_html.index(
        f"Unlimited-OCR pages: {newest_source}"
    ) < live_html.index("Unlimited-OCR files (test)")
    assert "1/2" in live_html
    assert "<progress" in live_html
    assert "Unlimited-OCR files (newest to oldest, 2 shown)" in final_html
    assert "Unlimited-OCR pages: finished" in final_html
    assert '<progress value="1" max="1"' in final_html
    assert '<progress style="' not in final_html
    assert "<strong>processed</strong>" in final_html
    assert "<strong>skipped</strong>" in final_html
    assert final_html.index(str(newest_source)) < final_html.index(str(older_source))
    assert "elapsed=0.25s" in final_html


async def test_folder_reports_page_progress_and_all_files():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        pdf_path = root / "a-new.pdf"
        skipped_source = root / "b-skipped.png"
        make_pdf(pdf_path, labels=("one", "two"))
        make_image(skipped_source)
        output_root = root / ".md_unlimited"
        output_root.mkdir()
        (output_root / "b-skipped.md").write_text(
            "existing", encoding="utf-8"
        )
        client = FakeAsyncOpenAIClient(("First", "Second"))
        output = StringIO()
        progress_output = StringIO()
        with (
            patch(__name__ + "._running_in_notebook", return_value=False),
            redirect_stdout(output),
            redirect_stderr(progress_output),
        ):
            results = await ocr_folder(
                root,
                client=client,
                max_concurrency=1,
                page_concurrency=2,
            )

        test_eq(
            [result.status for result in results],
            ["processed", "skipped"],
        )
        rendered_progress = progress_output.getvalue()
        assert f"Unlimited-OCR pages: {pdf_path.resolve()}" in rendered_progress
        assert "2/2" in rendered_progress
        assert "Unlimited-OCR files (http://172.27.74.16:7870/v1" in (
            rendered_progress
        )
        report = output.getvalue()
        assert f"Unlimited-OCR file: {pdf_path.resolve()} | status=processed" in report
        assert (
            f"Unlimited-OCR file: {skipped_source.resolve()} | status=skipped"
            in report
        )


await test_office_sources_convert_and_follow_pdf_ocr_path()
await test_image_skip_failure_and_folder_concurrency()
test_notebook_progress_renders_page_file_and_history_rows()
await test_folder_reports_page_progress_and_all_files()

In [ ]:
# | hide
async def test_adaptive_pdf_layout_rendering():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "adaptive-layout.pdf"
        target = root / "adaptive-layout.md"
        document = pymupdf.open()
        document.new_page(width=1000, height=1000).insert_text((10, 50), "large")
        document.save(source)
        document.close()
        detector = FakeLayoutDetector(
            [
                {
                    "index": 0,
                    "label": "text",
                    "score": 0.99,
                    "bbox_2d": [0, 0, 1000, 1000],
                    "task_type": "text",
                }
            ]
        )

        result = await ocr_pdf(
            source,
            target,
            client=FakeAsyncOpenAIClient(("recognized",)),
            dpi=72,
            max_page_pixels=100_000,
            page_concurrency=1,
            layout_mode="pp-doclayout",
            layout_detector=detector,
        )

        assert result.status == "processed", result.error
        layout = json.loads(
            target.with_suffix(".layout.json").read_text(encoding="utf-8")
        )
        test_eq(layout["settings"]["max_page_pixels"], 100_000)
        assert layout["pages"][0]["effective_dpi"] < 72
        assert layout["pages"][0]["render_pixels"] <= 100_000
        test_eq(layout["pages"][0]["render_warnings"], [])

    render_calls: list[dict[str, Any]] = []
    expected_pixmap = pymupdf.Pixmap(pymupdf.csRGB, 1, 1, b"\xff\xff\xff", False)

    def get_pixmap(**kwargs):
        render_calls.append(kwargs)
        return expected_pixmap

    fake_page = SimpleNamespace(rect=pymupdf.Rect(0, 0, 72, 72), get_pixmap=get_pixmap)
    rendered, effective_dpi, render_warnings = _render_pdf_page(
        fake_page,
        dpi=72,
    )
    assert rendered is expected_pixmap
    test_eq(effective_dpi, 72)
    test_eq(render_warnings, ())
    test_eq(render_calls, [{"dpi": 72, "alpha": False, "annots": False}])
    load_calls: list[int] = []

    def load_page(page_index):
        load_calls.append(page_index)
        return fake_page

    loaded, load_warnings = _load_pdf_page(
        SimpleNamespace(load_page=load_page),
        3,
    )
    assert loaded is fake_page
    test_eq(load_warnings, ())
    test_eq(load_calls, [3])


await test_adaptive_pdf_layout_rendering()


async def test_structured_layout_reconstruction_and_partial_bundle():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "structured.png"
        target = root / "structured.md"
        make_image(source, "structured")
        detector = FakeLayoutDetector(
            [
                {
                    "index": 0,
                    "label": "doc_title",
                    "score": 0.99,
                    "bbox_2d": [0, 0, 1000, 250],
                    "task_type": "text",
                },
                {
                    "index": 1,
                    "label": "table",
                    "score": 0.98,
                    "bbox_2d": [0, 250, 500, 1000],
                    "task_type": "table",
                },
                {
                    "index": 2,
                    "label": "image",
                    "score": 0.97,
                    "bbox_2d": [500, 250, 1000, 1000],
                    "task_type": "figure",
                },
            ]
        )
        raw_title = (
            "<|ref|>SCARA Robot<|/ref|>"
            "<|det|>[[0,0,999,250]]<|/det|>"
        )
        table = "<table><tr><td>10 kg</td></tr></table>"
        client = FakeAsyncOpenAIClient((raw_title, table))
        result = await ocr_image(
            source,
            target,
            client=client,
            layout_mode="pp-doclayout",
            layout_detector=detector,
            embed_page_image=True,
        )

        assert result.status == "processed", result.error
        test_eq(
            (result.regions_total, result.regions_completed, result.regions_failed),
            (3, 3, 0),
        )
        test_eq(len(client.calls), 2)
        test_eq([call["max_tokens"] for call in client.calls], [2048, 8192])
        for call in client.calls:
            test_eq(
                call["messages"][0]["content"][0],
                {"type": "text", "text": "<image>document parsing."},
            )
            test_eq(
                call["extra_body"],
                {
                    "skip_special_tokens": False,
                    "vllm_xargs": {"ngram_size": 35, "window_size": 128},
                },
            )

        markdown = target.read_text(encoding="utf-8")
        assert "# SCARA Robot" in markdown
        assert table in markdown
        assert (
            "![Table 2](<structured.assets/"
            "page-0001-region-002-table.png>)"
        ) in markdown
        assert (
            "![Image 3](<structured.assets/"
            "page-0001-region-003-image.png>)"
        ) in markdown
        assert "![Original page 1](<structured.assets/page-0001.png>)" in markdown
        assets = sorted((root / "structured.assets").glob("*.png"))
        test_eq(len(assets), 3)
        layout = json.loads(
            target.with_suffix(".layout.json").read_text(encoding="utf-8")
        )
        test_eq(layout["schema_version"], 2)
        test_eq(layout["provider"], "baidu/Unlimited-OCR")
        test_eq(
            layout["settings"]["response_validation_version"],
            _LAYOUT_RESPONSE_VALIDATION_VERSION,
        )
        test_eq(
            [region["status"] for region in layout["pages"][0]["regions"]],
            ["completed", "completed", "preserved"],
        )
        test_eq(
            [region["max_tokens"] for region in layout["pages"][0]["regions"]],
            [2048, 8192, None],
        )
        test_eq(layout["pages"][0]["regions"][0]["raw_content"], raw_title)
        assert "api_key" not in json.dumps(layout)
        assert detector.calls[0][0][0].size == (240, 120)
        assert not detector.stopped
        assert not (root / "structured.partial.md").exists()
        assert not (root / ".structured.layout-work").exists()

        failed_target = root / "partial.md"
        failed_detector = FakeLayoutDetector(
            [
                {
                    "index": 0,
                    "label": "text",
                    "score": 0.9,
                    "bbox_2d": [0, 0, 500, 1000],
                    "task_type": "text",
                },
                {
                    "index": 1,
                    "label": "table",
                    "score": 0.9,
                    "bbox_2d": [500, 0, 1000, 1000],
                    "task_type": "table",
                },
            ]
        )
        partial = await ocr_image(
            source,
            failed_target,
            client=FakeAsyncOpenAIClient(("recognized", RuntimeError("region failed"))),
            layout_mode="pp-doclayout",
            layout_detector=failed_detector,
        )
        test_eq(partial.status, "partial")
        test_eq(
            (
                partial.regions_total,
                partial.regions_completed,
                partial.regions_failed,
            ),
            (2, 1, 1),
        )
        partial_markdown = failed_target.read_text(encoding="utf-8")
        assert "OCR failed for `table` region 2" in partial_markdown
        assert "![Table 2](<partial.assets/page-0001-region-002-table.png>)" in (
            partial_markdown
        )
        failed_layout = json.loads(
            failed_target.with_suffix(".layout.json").read_text(encoding="utf-8")
        )
        test_eq(failed_layout["status"], "partial")
        test_eq(failed_layout["pages"][0]["regions"][1]["status"], "failed")
        partial_checkpoint = root / ".partial.layout-work"
        assert (root / "partial.partial.md").is_file()
        assert partial_checkpoint.is_dir()

        # Simulate a legacy published partial whose work directory was removed.
        shutil.rmtree(partial_checkpoint)
        (root / "partial.partial.md").unlink()
        (root / "partial.partial.layout.json").unlink()
        resumed_client = FakeAsyncOpenAIClient(("recovered table",))
        resumed = await ocr_image(
            source,
            failed_target,
            client=resumed_client,
            layout_mode="pp-doclayout",
            layout_detector=failed_detector,
        )
        assert resumed.status == "processed", resumed.error
        test_eq(len(resumed_client.calls), 1)
        assert "recovered table" in failed_target.read_text(encoding="utf-8")
        assert not (root / "partial.partial.md").exists()
        assert not partial_checkpoint.exists()


await test_structured_layout_reconstruction_and_partial_bundle()


async def test_empty_layout_output_retry_and_preservation():
    empty_grounding = "<|det|>[[0,0,999,999]]<|/det|>"
    regions = [
        {
            "index": 0,
            "label": "paragraph_title",
            "score": 0.99,
            "bbox_2d": [0, 0, 1000, 150],
            "task_type": "text",
        }
    ]
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "retry.png"
        make_image(source, "wide title")

        recovered_target = root / "recovered.md"
        recovered_client = FakeAsyncOpenAIClient(
            (empty_grounding, "Recovered title")
        )
        recovered = await ocr_image(
            source,
            recovered_target,
            client=recovered_client,
            layout_mode="pp-doclayout",
            layout_detector=FakeLayoutDetector(regions),
        )
        assert recovered.status == "processed", recovered.error
        test_eq(len(recovered_client.calls), 2)
        recovered_layout = json.loads(
            recovered_target.with_suffix(".layout.json").read_text(encoding="utf-8")
        )
        recovered_region = recovered_layout["pages"][0]["regions"][0]
        test_eq(recovered_region["status"], "recovered")
        assert "aspect-ratio-padded" in recovered_region["recovery"]

        preserved_target = root / "preserved.md"
        preserved = await ocr_image(
            source,
            preserved_target,
            client=FakeAsyncOpenAIClient((empty_grounding, empty_grounding)),
            layout_mode="pp-doclayout",
            layout_detector=FakeLayoutDetector(regions),
        )
        assert preserved.status == "processed", preserved.error
        test_eq((preserved.regions_completed, preserved.regions_failed), (1, 0))
        preserved_layout = json.loads(
            preserved_target.with_suffix(".layout.json").read_text(encoding="utf-8")
        )
        test_eq(preserved_layout["pages"][0]["regions"][0]["status"], "preserved")
        preserved_markdown = preserved_target.read_text(encoding="utf-8")
        assert "![Preserved paragraph_title region 1]" in preserved_markdown
        assert "no usable text after a contextual retry" in preserved_markdown


await test_empty_layout_output_retry_and_preservation()

In [ ]:
# | hide
async def test_layout_checkpoint_resume_and_settings_guard():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "resume.png"
        target = root / "resume.md"
        make_image(source)
        detector = FakeLayoutDetector(
            [
                {
                    "index": 0,
                    "label": "text",
                    "score": 0.9,
                    "bbox_2d": [0, 0, 500, 1000],
                    "task_type": "text",
                },
                {
                    "index": 1,
                    "label": "text",
                    "score": 0.9,
                    "bbox_2d": [500, 0, 1000, 1000],
                    "task_type": "text",
                },
            ]
        )
        task = asyncio.create_task(
            ocr_image(
                source,
                target,
                client=FakeAsyncOpenAIClient(
                    ((0.01, "First"), (2.0, "Second"))
                ),
                layout_mode="pp-doclayout",
                layout_detector=detector,
            )
        )
        checkpoint_path = root / ".resume.layout-work" / "checkpoint.json"
        partial_path = root / "resume.partial.md"
        for _ in range(200):
            if checkpoint_path.is_file():
                checkpoint = json.loads(checkpoint_path.read_text(encoding="utf-8"))
                if _layout_counts(checkpoint)[1] == 1:
                    break
            await asyncio.sleep(0.01)
        else:
            raise AssertionError("First region was not checkpointed")
        assert partial_path.is_file()
        assert "OCR processing is incomplete" in partial_path.read_text(
            encoding="utf-8"
        )
        task.cancel()
        try:
            await task
        except asyncio.CancelledError:
            pass

        resumed_client = FakeAsyncOpenAIClient(("Second",))
        resumed = await ocr_image(
            source,
            target,
            client=resumed_client,
            layout_mode="pp-doclayout",
            layout_detector=detector,
        )
        assert resumed.status == "processed", resumed.error
        test_eq(len(resumed_client.calls), 1)
        final_markdown = target.read_text(encoding="utf-8")
        assert "First" in final_markdown and "Second" in final_markdown
        test_eq(len(detector.calls), 1)
        assert not partial_path.exists()
        assert not checkpoint_path.parent.exists()

    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "settings.png"
        target = root / "settings.md"
        make_image(source)
        settings = _layout_settings(
            service_base_url="http://172.27.74.16:7870/v1",
            model="baidu/Unlimited-OCR",
            prompt="<image>document parsing.",
            dpi=None,
            layout_model=_DEFAULT_LAYOUT_MODEL,
            layout_threshold=0.3,
            embed_page_image=False,
            max_tokens=8192,
            ngram_size=35,
            ngram_window=128,
            max_data_url_bytes=_DEFAULT_MAX_DATA_URL_BYTES,
            clean_output=True,
        )
        _, _, original = _prepare_layout_workspace(
            source,
            target,
            pages_total=1,
            settings=settings,
            resume_partial=True,
            overwrite=False,
        )
        changed = {**settings, "max_tokens": 4096}
        try:
            _prepare_layout_workspace(
                source,
                target,
                pages_total=1,
                settings=changed,
                resume_partial=True,
                overwrite=False,
            )
        except ValueError as error:
            assert "incompatible" in str(error).casefold()
        else:
            raise AssertionError("Expected an incompatible checkpoint")
        _, _, replaced = _prepare_layout_workspace(
            source,
            target,
            pages_total=1,
            settings=changed,
            resume_partial=True,
            overwrite=True,
        )
        test_eq(original["settings"]["max_tokens"], 8192)
        test_eq(replaced["settings"]["max_tokens"], 4096)

    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "legacy.pdf"
        target = root / "legacy.md"
        source.touch()
        legacy_settings = _layout_settings(
            service_base_url="http://172.27.74.16:7870/v1",
            model="baidu/Unlimited-OCR",
            prompt="<image>document parsing.",
            dpi=300,
            layout_model=_DEFAULT_LAYOUT_MODEL,
            layout_threshold=0.3,
            embed_page_image=False,
            max_tokens=8192,
            ngram_size=35,
            ngram_window=128,
            max_data_url_bytes=_DEFAULT_MAX_DATA_URL_BYTES,
            clean_output=True,
        )
        _, assets, legacy = _prepare_layout_workspace(
            source,
            target,
            pages_total=2,
            settings=legacy_settings,
            resume_partial=True,
            overwrite=False,
        )
        legacy["pages"] = [
            {"page_number": 1, "status": "completed", "regions": []},
            {"page_number": 2, "status": "processing", "regions": []},
        ]
        completed_asset = assets / "page-0001-region-001-text.png"
        incomplete_asset = assets / "page-0002-region-001-text.png"
        completed_asset.write_bytes(b"completed")
        incomplete_asset.write_bytes(b"incomplete")
        _write_layout_checkpoint(target, legacy)

        adaptive_settings = {
            **legacy_settings,
            "max_page_pixels": 100_000,
        }
        _, _, migrated = _prepare_layout_workspace(
            source,
            target,
            pages_total=2,
            settings=adaptive_settings,
            resume_partial=True,
            overwrite=False,
        )
        test_eq(migrated["settings"], adaptive_settings)
        test_eq(migrated["pages"][0]["status"], "completed")
        assert migrated["pages"][1] is None
        assert completed_asset.is_file()
        assert not incomplete_asset.exists()


await test_layout_checkpoint_resume_and_settings_guard()

In [ ]:
# | hide
def test_layout_publication_rolls_back():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        stage = root / "stage"
        stage.mkdir()
        target = root / "document.md"
        sidecar = root / "document.layout.json"
        assets = root / "document.assets"
        target.write_text("old markdown", encoding="utf-8")
        sidecar.write_text("old sidecar", encoding="utf-8")
        assets.mkdir()
        (assets / "old.png").write_bytes(b"old")
        staged_assets = stage / "document.assets"
        staged_assets.mkdir()
        (staged_assets / "new.png").write_bytes(b"new")
        original_replace = Path.replace

        def fail_on_sidecar(path, destination):
            if path == stage / "document.layout.json":
                raise OSError("simulated publication failure")
            return original_replace(path, destination)

        with patch.object(Path, "replace", fail_on_sidecar):
            try:
                _publish_layout_bundle(
                    stage,
                    target,
                    "new markdown",
                    {"schema_version": 2},
                )
            except OSError as error:
                assert "simulated publication failure" in str(error)
            else:
                raise AssertionError("Expected publication failure")
        test_eq(target.read_text(encoding="utf-8"), "old markdown")
        test_eq(sidecar.read_text(encoding="utf-8"), "old sidecar")
        assert (assets / "old.png").is_file()
        assert not (assets / "new.png").exists()


test_layout_publication_rolls_back()

In [ ]:
# | hide
async def test_folder_layout_detector_reuse_and_output_lock():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_image(root / "a.png")
        make_image(root / "b.png")
        detector = FakeLayoutDetector(
            [
                {
                    "index": 0,
                    "label": "text",
                    "score": 0.9,
                    "bbox_2d": [0, 0, 1000, 1000],
                    "task_type": "text",
                }
            ]
        )
        output = StringIO()
        with redirect_stdout(output):
            results = await ocr_folder(
                root,
                client=FakeAsyncOpenAIClient(("A", "B")),
                layout_mode="pp-doclayout",
                layout_detector=detector,
                max_concurrency=2,
                show_progress=False,
            )
        test_eq([result.status for result in results], ["processed", "processed"])
        test_eq(len(detector.calls), 2)
        assert (root / ".md_unlimited" / "a.layout.json").is_file()
        assert (root / ".md_unlimited" / "b.layout.json").is_file()
        assert not detector.stopped
        assert "0 partial" in output.getvalue()

    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_image(root / "locked.png")
        lock_path = root / ".unlimited-ocr-folder.lock"
        with _exclusive_output_lock(lock_path, "test folder"):
            try:
                await ocr_folder(
                    root,
                    client=FakeAsyncOpenAIClient(("unused",)),
                    show_progress=False,
                )
            except RuntimeError as error:
                assert "another ocr process" in str(error).casefold()
            else:
                raise AssertionError("Expected the folder output lock to reject")


await test_folder_layout_detector_reuse_and_output_lock()

In [ ]:
# | hide
async def test_invalid_inputs():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        corrupt = root / "corrupt.pdf"
        corrupt.write_bytes(b"not a PDF")
        corrupt_result = await ocr_pdf(
            corrupt,
            root / "corrupt.md",
            client=FakeAsyncOpenAIClient(),
        )
        test_eq(corrupt_result.status, "failed")

        encrypted = root / "encrypted.pdf"
        make_pdf(encrypted, password="secret")
        encrypted_result = await ocr_pdf(
            encrypted,
            root / "encrypted.md",
            client=FakeAsyncOpenAIClient(),
        )
        test_eq(encrypted_result.status, "failed")
        assert "password" in encrypted_result.error.casefold()

        image = root / "empty.png"
        make_image(image)
        empty_response = SimpleNamespace(
            choices=[
                SimpleNamespace(
                    message=SimpleNamespace(content="  "),
                    finish_reason="stop",
                )
            ]
        )
        empty_result = await ocr_image(
            image,
            root / "empty.md",
            client=FakeAsyncOpenAIClient((empty_response,)),
        )
        test_eq(empty_result.status, "failed")

        for kwargs, expected in (
            ({"dpi": 0}, "dpi"),
            ({"max_page_pixels": 0}, "max_page_pixels"),
            ({"office_conversion_timeout_s": 0}, "office_conversion_timeout"),
            ({"request_timeout_s": 0}, "request_timeout"),
            ({"max_concurrency": 0}, "MAX_CONCURRENCY"),
            ({"page_concurrency": 0}, "PAGE_CONCURRENCY"),
            ({"output_dir_name": "a/b"}, "output_dir_name"),
            ({"prompt": "document parsing."}, "<image>"),
            ({"model": " "}, "model"),
            ({"max_tokens": 0}, "max_tokens"),
            ({"layout_mode": "invalid"}, "layout_mode"),
            (
                {"layout_mode": "pp-doclayout", "layout_threshold": 1.1},
                "layout_threshold",
            ),
            (
                {"layout_mode": "pp-doclayout", "layout_model": " "},
                "layout_model",
            ),
        ):
            try:
                await ocr_folder(root, client=FakeAsyncOpenAIClient(), **kwargs)
            except Exception as error:
                assert expected.casefold() in str(error).casefold()
            else:
                raise AssertionError(f"Expected failure containing {expected!r}")


await test_invalid_inputs()